## Step 29 — User & Merchant Velocity and Repeated Activity Analysis

In [1]:
import pandas as pd
import numpy as np

# Load the final enriched transaction fact table
fact_transactions = pd.read_csv(
    "../data/processed/fact_transactions_enriched.csv"
)

print("Fact table shape:", fact_transactions.shape)
print("\nColumns:")
print(fact_transactions.columns.tolist())

Fact table shape: (20000, 78)

Columns:
['txn_id', 'timestamp', 'user_id', 'merchant_id', 'amount', 'utr', 'mcc', 'status', 'timestamp_original', 'transaction_date', 'transaction_hour', 'transaction_day', 'transaction_month', 'transaction_year', 'mcc_original', 'mcc_missing_flag', 'kyc_match_flag', 'merchant_match_flag', 'full_name', 'pan', 'aadhaar', 'date_of_birth', 'city', 'state', 'monthly_income', 'occupation', 'kyc_status', 'risk_segment', 'merchant_name', 'mcc_merchant', 'merchant_category', 'business_type', 'city_merchant', 'state_merchant', 'onboarding_date', 'settlement_account', 'merchant_status', 'declared_avg_ticket_size', 'user_id_missing_flag', 'merchant_id_missing_flag', 'chargeback_count', 'disputed_amount_total', 'fraud_unauthorized_chargeback_count', 'negative_reporting_delay_count', 'negative_bank_response_delay_count', 'max_chargeback_severity', 'chargeback_flag', 'fraud_chargeback_flag', 'amount_numeric', 'status_clean', 'successful_transaction_flag', 'failed_tran

In [2]:
# Prepare date and timestamp fields for velocity analysis

fact_transactions["timestamp_clean"] = pd.to_datetime(
    fact_transactions["timestamp_clean"],
    errors="coerce"
)

fact_transactions["transaction_date"] = pd.to_datetime(
    fact_transactions["transaction_date"],
    errors="coerce"
).dt.date

required_cols = [
    "txn_id",
    "user_id",
    "merchant_id",
    "timestamp_clean",
    "transaction_date"
]

print("Required columns check:")
for col in required_cols:
    print(f"{col}: {'✓' if col in fact_transactions.columns else '✗'}")

print("\nMissing values:")
print(fact_transactions[required_cols].isna().sum())

Required columns check:
txn_id: ✓
user_id: ✓
merchant_id: ✓
timestamp_clean: ✓
transaction_date: ✓

Missing values:
txn_id              0
user_id             0
merchant_id         0
timestamp_clean     0
transaction_date    0
dtype: int64


In [3]:
# Calculate daily transaction counts for each user and merchant

fact_transactions["user_daily_txn_count"] = (
    fact_transactions
    .groupby(["user_id", "transaction_date"])["txn_id"]
    .transform("count")
)

fact_transactions["merchant_daily_txn_count"] = (
    fact_transactions
    .groupby(["merchant_id", "transaction_date"])["txn_id"]
    .transform("count")
)

print("User daily transaction count:")
print(fact_transactions["user_daily_txn_count"].describe())

print("\nMerchant daily transaction count:")
print(fact_transactions["merchant_daily_txn_count"].describe())

print("\nMaximum user transactions in one day:",
      fact_transactions["user_daily_txn_count"].max())

print("Maximum merchant transactions in one day:",
      fact_transactions["merchant_daily_txn_count"].max())

User daily transaction count:
count    20000.000000
mean         1.001800
std          0.042389
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          2.000000
Name: user_daily_txn_count, dtype: float64

Merchant daily transaction count:
count    20000.000000
mean         1.021500
std          0.147101
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          3.000000
Name: merchant_daily_txn_count, dtype: float64

Maximum user transactions in one day: 2
Maximum merchant transactions in one day: 3


In [4]:
# Calculate data-driven velocity thresholds

user_velocity_threshold = fact_transactions["user_daily_txn_count"].quantile(0.95)
merchant_velocity_threshold = fact_transactions["merchant_daily_txn_count"].quantile(0.95)

print("95th percentile user daily transaction count:",
      user_velocity_threshold)

print("95th percentile merchant daily transaction count:",
      merchant_velocity_threshold)

# Create velocity flags
fact_transactions["high_user_velocity_flag"] = (
    fact_transactions["user_daily_txn_count"] > user_velocity_threshold
).astype(int)

fact_transactions["high_merchant_velocity_flag"] = (
    fact_transactions["merchant_daily_txn_count"] > merchant_velocity_threshold
).astype(int)

fact_transactions["velocity_anomaly_flag"] = (
    (fact_transactions["high_user_velocity_flag"] == 1) |
    (fact_transactions["high_merchant_velocity_flag"] == 1)
).astype(int)

print("\nHigh user velocity transactions:",
      fact_transactions["high_user_velocity_flag"].sum())

print("High merchant velocity transactions:",
      fact_transactions["high_merchant_velocity_flag"].sum())

print("Total velocity anomaly transactions:",
      fact_transactions["velocity_anomaly_flag"].sum())

95th percentile user daily transaction count: 1.0
95th percentile merchant daily transaction count: 1.0

High user velocity transactions: 36
High merchant velocity transactions: 424
Total velocity anomaly transactions: 460


In [5]:
# Calculate repeated chargeback activity by user and merchant

# Number of chargebacks associated with each user
user_chargeback_counts = (
    fact_transactions
    .groupby("user_id")["chargeback_count"]
    .sum()
)

fact_transactions["user_chargeback_count"] = (
    fact_transactions["user_id"]
    .map(user_chargeback_counts)
    .fillna(0)
)

# Flag users with 2 or more chargebacks
fact_transactions["repeated_chargeback_user_flag"] = (
    fact_transactions["user_chargeback_count"] >= 2
).astype(int)


# Number of chargebacks associated with each merchant
merchant_chargeback_counts = (
    fact_transactions
    .groupby("merchant_id")["chargeback_count"]
    .sum()
)

fact_transactions["merchant_chargeback_count"] = (
    fact_transactions["merchant_id"]
    .map(merchant_chargeback_counts)
    .fillna(0)
)

# Data-driven merchant threshold
merchant_chargeback_threshold = (
    fact_transactions["merchant_chargeback_count"]
    .quantile(0.95)
)

fact_transactions["high_chargeback_merchant_flag"] = (
    fact_transactions["merchant_chargeback_count"]
    > merchant_chargeback_threshold
).astype(int)


print("Users with 2 or more chargebacks:",
      fact_transactions["repeated_chargeback_user_flag"].sum())

print("\n95th percentile merchant chargeback count:",
      merchant_chargeback_threshold)

print("High-chargeback merchant transactions:",
      fact_transactions["high_chargeback_merchant_flag"].sum())

Users with 2 or more chargebacks: 266

95th percentile merchant chargeback count: 2.0
High-chargeback merchant transactions: 242


In [6]:
# Top users by chargeback count
top_users = (
    fact_transactions[
        fact_transactions["user_chargeback_count"] > 0
    ][["user_id", "user_chargeback_count"]]
    .drop_duplicates()
    .sort_values("user_chargeback_count", ascending=False)
    .head(10)
)

print("Top 10 users by chargeback count:")
display(top_users)


# Top merchants by chargeback count
top_merchants = (
    fact_transactions[
        fact_transactions["merchant_chargeback_count"] > 0
    ][["merchant_id", "merchant_name", "merchant_category",
       "merchant_chargeback_count"]]
    .drop_duplicates()
    .sort_values("merchant_chargeback_count", ascending=False)
    .head(10)
)

print("\nTop 10 merchants by chargeback count:")
display(top_merchants)


# Unique high-chargeback merchants
high_chargeback_merchants = (
    fact_transactions.loc[
        fact_transactions["high_chargeback_merchant_flag"] == 1,
        "merchant_id"
    ]
    .nunique()
)

print(
    "\nUnique high-chargeback merchants:",
    high_chargeback_merchants
)

Top 10 users by chargeback count:


,user_id,user_chargeback_count
3499,USR58627,4
7063,USR96715,3
4448,USR60393,3
2231,USR63885,3
1572,USR58522,3
5437,USR88654,3
8308,USR30575,3
9076,USR34733,3
10385,USR48734,3
15102,USR95217,3



Top 10 merchants by chargeback count:


,merchant_id,merchant_name,merchant_category,merchant_chargeback_count
5720,MCH9572,NaN,NaN,5
2208,MCH5530,NaN,NaN,4
1050,MCH3587,Sagar-Raval,HOTEL_LODGING,4
2483,MCH6701,NaN,NaN,3
791,MCH1744,"Mani, Tara and Mane",TRAVEL,3
729,MCH5542,NaN,NaN,3
623,MCH6973,NaN,NaN,3
2578,MCH9431,Madan-Kota,TRAVEL,3
2846,MCH9675,Mand-Varkey,Medical,3
509,MCH6924,NaN,NaN,3



Unique high-chargeback merchants: 58


In [7]:
# Velocity distribution summary

user_velocity_summary = (
    fact_transactions["user_daily_txn_count"]
    .value_counts()
    .sort_index()
)

merchant_velocity_summary = (
    fact_transactions["merchant_daily_txn_count"]
    .value_counts()
    .sort_index()
)

print("User daily transaction velocity:")
print(user_velocity_summary)

print("\nMerchant daily transaction velocity:")
print(merchant_velocity_summary)

print("\nVelocity anomaly summary:")
print(
    fact_transactions["velocity_anomaly_flag"]
    .value_counts()
    .rename(index={0: "Normal", 1: "Anomaly"})
)

User daily transaction velocity:
user_daily_txn_count
1    19964
2       36
Name: count, dtype: int64

Merchant daily transaction velocity:
merchant_daily_txn_count
1    19576
2      418
3        6
Name: count, dtype: int64

Velocity anomaly summary:
velocity_anomaly_flag
Normal     19540
Anomaly      460
Name: count, dtype: int64


In [8]:
# Save the enriched fact table with Step 29 velocity and repeated-activity features

fact_transactions.to_csv(
    "../data/processed/fact_transactions_enriched.csv",
    index=False
)

print("Fact table saved successfully.")
print("Shape:", fact_transactions.shape)

print("\nNew Step 29 columns:")
step29_cols = [
    "user_daily_txn_count",
    "merchant_daily_txn_count",
    "high_user_velocity_flag",
    "high_merchant_velocity_flag",
    "velocity_anomaly_flag",
    "user_chargeback_count",
    "repeated_chargeback_user_flag",
    "merchant_chargeback_count",
    "high_chargeback_merchant_flag"
]

print(step29_cols)

Fact table saved successfully.
Shape: (20000, 78)

New Step 29 columns:
['user_daily_txn_count', 'merchant_daily_txn_count', 'high_user_velocity_flag', 'high_merchant_velocity_flag', 'velocity_anomaly_flag', 'user_chargeback_count', 'repeated_chargeback_user_flag', 'merchant_chargeback_count', 'high_chargeback_merchant_flag']


In [9]:
# Load cleaned KYC data and recreate the identity profile

kyc_clean = pd.read_csv(
    "../data/processed/kyc_clean.csv"
)

print("Cleaned KYC shape:", kyc_clean.shape)

print("\nKYC columns:")
print(kyc_clean.columns.tolist())

Cleaned KYC shape: (36122, 21)

KYC columns:
['user_id', 'full_name', 'pan', 'aadhaar', 'date_of_birth', 'city', 'state', 'monthly_income', 'occupation', 'signup_timestamp', 'kyc_status', 'risk_segment', 'kyc_status_original', 'pan_missing_flag', 'aadhaar_missing_flag', 'dob_missing_flag', 'income_missing_flag', 'signup_missing_flag', 'kyc_rejected_flag', 'kyc_pending_flag', 'high_risk_segment_flag']


In [10]:
# Recreate user-level identity profile

identity_profile = (
    kyc_clean
    .groupby("user_id")
    .agg(
        full_name=("full_name", "nunique"),
        pan=("pan", "nunique"),
        aadhaar=("aadhaar", "nunique"),
        date_of_birth=("date_of_birth", "nunique"),
        city=("city", "nunique"),
        state=("state", "nunique"),
        occupation=("occupation", "nunique"),
        identity_rows=("user_id", "size"),
        pan_missing=("pan_missing_flag", "sum"),
        aadhaar_missing=("aadhaar_missing_flag", "sum"),
        dob_missing=("dob_missing_flag", "sum"),
        kyc_rejected=("kyc_rejected_flag", "sum"),
    )
    .reset_index()
)

# Potential identity conflict:
# More than one distinct value for a core identity attribute
identity_profile["identity_conflict_flag"] = (
    (identity_profile["full_name"] > 1) |
    (identity_profile["pan"] > 1) |
    (identity_profile["aadhaar"] > 1) |
    (identity_profile["date_of_birth"] > 1)
).astype(int)

# Count the number of conflicting identity attributes
identity_profile["identity_conflict_count"] = (
    (identity_profile["full_name"] > 1).astype(int) +
    (identity_profile["pan"] > 1).astype(int) +
    (identity_profile["aadhaar"] > 1).astype(int) +
    (identity_profile["date_of_birth"] > 1).astype(int)
)

print("Identity profile shape:", identity_profile.shape)

print("\nPotential identity conflicts:")
print(
    identity_profile["identity_conflict_flag"]
    .value_counts()
    .rename(index={
        0: "No conflict",
        1: "Potential conflict"
    })
)

print("\nUsers with identity conflicts:",
      identity_profile["identity_conflict_flag"].sum())

Identity profile shape: (29355, 15)

Potential identity conflicts:
identity_conflict_flag
No conflict           24356
Potential conflict     4999
Name: count, dtype: int64

Users with identity conflicts: 4999


In [11]:
# Breakdown of identity conflict types

identity_conflict_summary = pd.DataFrame({
    "conflict_type": [
        "Multiple names",
        "Multiple PANs",
        "Multiple Aadhaar values",
        "Multiple dates of birth"
    ],
    "users_affected": [
        (identity_profile["full_name"] > 1).sum(),
        (identity_profile["pan"] > 1).sum(),
        (identity_profile["aadhaar"] > 1).sum(),
        (identity_profile["date_of_birth"] > 1).sum()
    ]
})

print("Identity conflict breakdown:")
display(identity_conflict_summary)

print("\nNumber of users by conflict count:")
print(
    identity_profile["identity_conflict_count"]
    .value_counts()
    .sort_index()
)

Identity conflict breakdown:


,conflict_type,users_affected
0,Multiple names,4999
1,Multiple PANs,4554
2,Multiple Aadhaar values,4378
3,Multiple dates of birth,4314



Number of users by conflict count:
identity_conflict_count
0    24356
1       14
2      195
3     1319
4     3471
Name: count, dtype: int64


In [12]:
# Create individual identity-risk signals

identity_profile["multiple_name_flag"] = (
    identity_profile["full_name"] > 1
).astype(int)

identity_profile["multiple_pan_flag"] = (
    identity_profile["pan"] > 1
).astype(int)

identity_profile["multiple_aadhaar_flag"] = (
    identity_profile["aadhaar"] > 1
).astype(int)

identity_profile["multiple_dob_flag"] = (
    identity_profile["date_of_birth"] > 1
).astype(int)

# Strong identity anomaly:
# conflicts across at least 3 core identity attributes
identity_profile["strong_identity_anomaly_flag"] = (
    identity_profile["identity_conflict_count"] >= 3
).astype(int)

print("Identity-risk signal counts:")
print(
    identity_profile[
        [
            "multiple_name_flag",
            "multiple_pan_flag",
            "multiple_aadhaar_flag",
            "multiple_dob_flag",
            "strong_identity_anomaly_flag"
        ]
    ].sum()
)

Identity-risk signal counts:
multiple_name_flag              4999
multiple_pan_flag               4554
multiple_aadhaar_flag           4378
multiple_dob_flag               4314
strong_identity_anomaly_flag    4790
dtype: int64


In [15]:
# Attach user-level identity risk signals to the transaction fact table

identity_features = identity_profile[
    [
        "user_id",
        "identity_conflict_flag",
        "identity_conflict_count",
        "multiple_name_flag",
        "multiple_pan_flag",
        "multiple_aadhaar_flag",
        "multiple_dob_flag",
        "strong_identity_anomaly_flag"
    ]
].copy()

# Ensure one row per user before merging
identity_features = identity_features.drop_duplicates("user_id")

# Merge identity features into transaction fact table
fact_transactions = fact_transactions.merge(
    identity_features,
    on="user_id",
    how="left"
)

# Users without a matching KYC identity profile
identity_columns = [
    "identity_conflict_flag",
    "identity_conflict_count",
    "multiple_name_flag",
    "multiple_pan_flag",
    "multiple_aadhaar_flag",
    "multiple_dob_flag",
    "strong_identity_anomaly_flag"
]

for col in identity_columns:
    fact_transactions[col] = fact_transactions[col].fillna(0)

# Validation
print("Fact table shape after identity merge:", fact_transactions.shape)
print("Unique transaction IDs:", fact_transactions["txn_id"].nunique())
print("Duplicate transaction rows:",
      fact_transactions["txn_id"].duplicated().sum())

print("\nIdentity anomaly transactions:",
      fact_transactions["identity_conflict_flag"].sum())

print("Strong identity anomaly transactions:",
      fact_transactions["strong_identity_anomaly_flag"].sum())

Fact table shape after identity merge: (20000, 92)
Unique transaction IDs: 20000
Duplicate transaction rows: 0

Identity anomaly transactions: 1181.0
Strong identity anomaly transactions: 1141.0


In [16]:
# Compare transaction behavior for identity-anomaly users

identity_behavior_summary = pd.DataFrame({
    "Metric": [
        "Total transactions",
        "Identity conflict transactions",
        "Strong identity anomaly transactions",
        "Chargeback transactions",
        "Fraud/unauthorized chargeback transactions",
        "High-value transactions",
        "Velocity anomaly transactions"
    ],
    "Count": [
        len(fact_transactions),
        fact_transactions["identity_conflict_flag"].sum(),
        fact_transactions["strong_identity_anomaly_flag"].sum(),
        fact_transactions["chargeback_flag"].sum(),
        fact_transactions["fraud_chargeback_flag"].sum(),
        fact_transactions["high_value_transaction_flag"].sum(),
        fact_transactions["velocity_anomaly_flag"].sum()
    ]
})

display(identity_behavior_summary)

,Metric,Count
0,Total transactions,20000.0
1,Identity conflict transactions,1181.0
2,Strong identity anomaly transactions,1141.0
3,Chargeback transactions,2451.0
4,Fraud/unauthorized chargeback transactions,873.0
5,High-value transactions,879.0
6,Velocity anomaly transactions,460.0


In [17]:
# Save fact table with identity-risk features

fact_transactions.to_csv(
    "../data/processed/fact_transactions_enriched.csv",
    index=False
)

print("Fact table saved successfully.")
print("Final shape:", fact_transactions.shape)

print("\nIdentity columns added:")
identity_cols = [
    "identity_conflict_flag",
    "identity_conflict_count",
    "multiple_name_flag",
    "multiple_pan_flag",
    "multiple_aadhaar_flag",
    "multiple_dob_flag",
    "strong_identity_anomaly_flag"
]

print(identity_cols)

Fact table saved successfully.
Final shape: (20000, 92)

Identity columns added:
['identity_conflict_flag', 'identity_conflict_count', 'multiple_name_flag', 'multiple_pan_flag', 'multiple_aadhaar_flag', 'multiple_dob_flag', 'strong_identity_anomaly_flag']


In [18]:
# Step 31: Merchant & Category Risk Analytics
# Category-level transaction and chargeback analysis

category_summary = (
    fact_transactions
    .groupby("merchant_category", dropna=False)
    .agg(
        transaction_count=("txn_id", "nunique"),
        transaction_value=("amount_numeric", "sum"),
        chargeback_transactions=("chargeback_flag", "sum"),
        fraud_chargeback_transactions=("fraud_chargeback_flag", "sum"),
        disputed_amount=("disputed_amount_total", "sum")
    )
    .reset_index()
)

# Calculate chargeback-to-transaction ratio
category_summary["chargeback_to_transaction_ratio"] = (
    category_summary["chargeback_transactions"] /
    category_summary["transaction_count"]
)

# Calculate fraud-chargeback ratio
category_summary["fraud_chargeback_ratio"] = (
    category_summary["fraud_chargeback_transactions"] /
    category_summary["transaction_count"]
)

# Sort by transaction volume
category_summary = category_summary.sort_values(
    "transaction_count",
    ascending=False
)

print("Number of merchant categories:",
      category_summary["merchant_category"].nunique())

display(category_summary)

Number of merchant categories: 65


,merchant_category,transaction_count,transaction_value,chargeback_transactions,fraud_chargeback_transactions,disputed_amount,chargeback_to_transaction_ratio,fraud_chargeback_ratio
65,NaN,10725,1.147821e+08,1303,476,4046529.17,0.121492,0.044382
30,Retail,285,2.994271e+06,39,12,120573.70,0.136842,0.042105
16,HOTEL_LODGING,267,2.968263e+06,29,10,88847.66,0.108614,0.037453
32,Stationery,255,2.971633e+06,36,12,139225.25,0.141176,0.047059
8,Department Store,254,2.955249e+06,22,11,56879.52,0.086614,0.043307
...,...,...,...,...,...,...,...,...
57,restaurant,3,2.460970e+04,0,0,0.00,0.000000,0.000000
58,restaurants,3,3.589950e+04,0,0,0.00,0.000000,0.000000
55,pharmacies,2,2.777883e+04,0,0,0.00,0.000000,0.000000
39,books_stationery,1,1.027532e+04,0,0,0.00,0.000000,0.000000


In [19]:
# Standardize merchant category labels

fact_transactions["merchant_category_clean"] = (
    fact_transactions["merchant_category"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Replace blank strings with missing values
fact_transactions["merchant_category_clean"] = (
    fact_transactions["merchant_category_clean"]
    .replace("", pd.NA)
)

print("Unique categories before standardization:",
      fact_transactions["merchant_category"].nunique(dropna=True))

print("Unique categories after standardization:",
      fact_transactions["merchant_category_clean"].nunique(dropna=True))

print("\nTop standardized merchant categories:")
display(
    fact_transactions["merchant_category_clean"]
    .value_counts(dropna=False)
    .head(20)
)

Unique categories before standardization: 65
Unique categories after standardization: 44

Top standardized merchant categories:


merchant_category_clean
<NA>                 10725
TELECOM                454
RETAIL                 289
HOTEL_LODGING          274
DEPARTMENT STORE       257
STATIONERY             256
MISCELLANEOUS          253
DEPT_STORE             253
MOBILE RECHARGE        246
PHONE SERVICE          240
DEPARTMENT STORES      239
CLOTHS                 237
BOOKS                  233
HOTEL                  222
TRANSPRT               215
TRANSPORT              214
BOOK STORE             214
RETAIL OTHER           213
KIRANA                 212
BOOKS_STATIONERY       210
Name: count, dtype: Int64

In [20]:
# Inspect all standardized merchant categories

all_categories = (
    fact_transactions["merchant_category_clean"]
    .dropna()
    .value_counts()
    .sort_index()
)

print("All standardized merchant categories:")
for category, count in all_categories.items():
    print(f"{category:<30} {count}")

print("\nTotal non-missing categories:", len(all_categories))

All standardized merchant categories:
APPAREL                        195
BOOK STORE                     214
BOOKS                          233
BOOKS_STATIONERY               210
BUS/TAXI                       198
CHEMIST                        163
CLOTHING                       176
CLOTHS                         237
DEPARTMENT STORE               257
DEPARTMENT STORES              239
DEPT_STORE                     253
EATING PLACE                   178
FASHION                        186
FOOD                           185
FOOD_SERVICES                  180
GARMENTS                       187
GROCERIES                      158
GROCERY                        174
GROCERY STORES                 162
GROCERY_STORE                  149
HOSPITALITY                    202
HOTEL                          222
HOTELS                         208
HOTEL_LODGING                  274
KIRANA                         212
MEDICAL                        158
MEDICAL_STORE                  208
MISC RETAIL      

In [21]:
# Consolidate clearly equivalent merchant-category labels

category_mapping = {
    # Books
    "BOOK STORE": "BOOKS",
    "BOOKS_STATIONERY": "BOOKS",

    # Department stores
    "DEPARTMENT STORES": "DEPARTMENT STORE",
    "DEPT_STORE": "DEPARTMENT STORE",

    # Grocery
    "GROCERIES": "GROCERY",
    "GROCERY STORES": "GROCERY",
    "GROCERY_STORE": "GROCERY",
    "KIRANA": "GROCERY",

    # Hotels / hospitality
    "HOTEL": "HOTEL_LODGING",
    "HOTELS": "HOTEL_LODGING",
    "HOSPITALITY": "HOTEL_LODGING",

    # Pharmacy
    "PHARMACIES": "PHARMACY",
    "CHEMIST": "PHARMACY",

    # Food
    "RESTAURANT": "FOOD_SERVICES",
    "RESTAURANTS": "FOOD_SERVICES",
    "EATING PLACE": "FOOD_SERVICES",
    "FOOD": "FOOD_SERVICES",

    # Apparel
    "CLOTHING": "APPAREL",
    "CLOTHS": "APPAREL",
    "FASHION": "APPAREL",
    "GARMENTS": "APPAREL",

    # Transport
    "TRANSPORTATION": "TRANSPORT",
    "TRANSPRT": "TRANSPORT",
    "BUS/TAXI": "TRANSPORT",

    # Miscellaneous retail
    "MISC RETAIL": "MISC_RETAIL",
    "MISCELLANEOUS": "MISC_RETAIL",
    "RETAIL OTHER": "MISC_RETAIL"
}

fact_transactions["merchant_category_final"] = (
    fact_transactions["merchant_category_clean"]
    .replace(category_mapping)
)

print("Categories before consolidation:",
      fact_transactions["merchant_category_clean"].nunique(dropna=True))

print("Categories after consolidation:",
      fact_transactions["merchant_category_final"].nunique(dropna=True))

print("\nFinal category distribution:")
display(
    fact_transactions["merchant_category_final"]
    .value_counts(dropna=False)
)

Categories before consolidation: 44
Categories after consolidation: 18

Final category distribution:


merchant_category_final
<NA>                10725
APPAREL               981
FOOD_SERVICES         918
HOTEL_LODGING         906
GROCERY               855
TRANSPORT             816
DEPARTMENT STORE      749
MISC_RETAIL           669
BOOKS                 657
PHARMACY              496
TELECOM               454
RETAIL                289
STATIONERY            256
MOBILE RECHARGE       246
PHONE SERVICE         240
MEDICAL_STORE         208
OTHER                 195
TRAVEL                182
MEDICAL               158
Name: count, dtype: Int64

In [22]:
# Calculate merchant-category risk metrics

category_risk = (
    fact_transactions
    .groupby("merchant_category_final", dropna=False)
    .agg(
        transaction_count=("txn_id", "nunique"),
        transaction_value=("amount_numeric", "sum"),
        chargeback_transactions=("chargeback_flag", "sum"),
        fraud_chargeback_transactions=("fraud_chargeback_flag", "sum"),
        disputed_amount=("disputed_amount_total", "sum")
    )
    .reset_index()
)

# Chargeback-to-transaction ratio
category_risk["chargeback_to_transaction_ratio"] = (
    category_risk["chargeback_transactions"] /
    category_risk["transaction_count"]
)

# Fraud/unauthorized chargeback ratio
category_risk["fraud_chargeback_ratio"] = (
    category_risk["fraud_chargeback_transactions"] /
    category_risk["transaction_count"]
)

# Sort by chargeback ratio
category_risk = category_risk.sort_values(
    "chargeback_to_transaction_ratio",
    ascending=False
)

print("Category risk summary:")
display(category_risk)

Category risk summary:


,merchant_category_final,transaction_count,transaction_value,chargeback_transactions,fraud_chargeback_transactions,disputed_amount,chargeback_to_transaction_ratio,fraud_chargeback_ratio
10,OTHER,195,2.217035e+06,35,11,131017.56,0.179487,0.056410
14,STATIONERY,256,2.979719e+06,36,12,139225.25,0.140625,0.046875
7,MEDICAL_STORE,208,2.128353e+06,29,11,112065.59,0.139423,0.052885
6,MEDICAL,158,1.774900e+06,22,8,70702.63,0.139241,0.050633
15,TELECOM,454,4.598871e+06,63,23,190165.63,0.138767,0.050661
13,RETAIL,289,3.040042e+06,40,12,121780.98,0.138408,0.041522
12,PHONE SERVICE,240,2.708886e+06,33,11,112240.85,0.137500,0.045833
16,TRANSPORT,816,8.571969e+06,111,36,296866.51,0.136029,0.044118
8,MISC_RETAIL,669,6.890983e+06,85,29,196894.50,0.127055,0.043348
17,TRAVEL,182,1.804988e+06,23,10,43858.00,0.126374,0.054945


In [23]:
# Validate category metrics for missing or invalid calculated values

print("Missing values in category risk metrics:")
print(
    category_risk[
        [
            "transaction_count",
            "transaction_value",
            "chargeback_transactions",
            "fraud_chargeback_transactions",
            "disputed_amount",
            "chargeback_to_transaction_ratio",
            "fraud_chargeback_ratio"
        ]
    ].isna().sum()
)

print("\nCategories with missing chargeback ratio:")
display(
    category_risk[
        category_risk["chargeback_to_transaction_ratio"].isna()
    ]
)

Missing values in category risk metrics:
transaction_count                  0
transaction_value                  0
chargeback_transactions            0
fraud_chargeback_transactions      0
disputed_amount                    0
chargeback_to_transaction_ratio    0
fraud_chargeback_ratio             0
dtype: int64

Categories with missing chargeback ratio:


,merchant_category_final,transaction_count,transaction_value,chargeback_transactions,fraud_chargeback_transactions,disputed_amount,chargeback_to_transaction_ratio,fraud_chargeback_ratio


In [24]:
# Create final ranked category-risk table

category_risk_ranked = category_risk[
    category_risk["merchant_category_final"].notna()
].copy()

# Rank categories by chargeback-to-transaction ratio
category_risk_ranked["chargeback_ratio_rank"] = (
    category_risk_ranked["chargeback_to_transaction_ratio"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

# Sort by risk ratio
category_risk_ranked = category_risk_ranked.sort_values(
    ["chargeback_to_transaction_ratio", "chargeback_transactions"],
    ascending=[False, False]
)

print("Top merchant categories by chargeback-to-transaction ratio:")

display(
    category_risk_ranked[
        [
            "chargeback_ratio_rank",
            "merchant_category_final",
            "transaction_count",
            "chargeback_transactions",
            "fraud_chargeback_transactions",
            "transaction_value",
            "disputed_amount",
            "chargeback_to_transaction_ratio",
            "fraud_chargeback_ratio"
        ]
    ].head(10)
)

Top merchant categories by chargeback-to-transaction ratio:


,chargeback_ratio_rank,merchant_category_final,transaction_count,chargeback_transactions,fraud_chargeback_transactions,transaction_value,disputed_amount,chargeback_to_transaction_ratio,fraud_chargeback_ratio
10,1,OTHER,195,35,11,2217035.07,131017.56,0.179487,0.056410
14,2,STATIONERY,256,36,12,2979719.27,139225.25,0.140625,0.046875
7,3,MEDICAL_STORE,208,29,11,2128353.04,112065.59,0.139423,0.052885
6,4,MEDICAL,158,22,8,1774900.50,70702.63,0.139241,0.050633
15,5,TELECOM,454,63,23,4598870.63,190165.63,0.138767,0.050661
13,6,RETAIL,289,40,12,3040042.12,121780.98,0.138408,0.041522
12,7,PHONE SERVICE,240,33,11,2708885.66,112240.85,0.137500,0.045833
16,8,TRANSPORT,816,111,36,8571969.26,296866.51,0.136029,0.044118
8,9,MISC_RETAIL,669,85,29,6890982.79,196894.50,0.127055,0.043348
17,10,TRAVEL,182,23,10,1804988.01,43858.00,0.126374,0.054945


In [25]:
# Merchant-level risk analysis

merchant_risk = (
    fact_transactions
    .groupby(
        ["merchant_id", "merchant_name", "merchant_category_final"],
        dropna=False
    )
    .agg(
        transaction_count=("txn_id", "nunique"),
        transaction_value=("amount_numeric", "sum"),
        chargeback_transactions=("chargeback_flag", "sum"),
        fraud_chargeback_transactions=("fraud_chargeback_flag", "sum"),
        disputed_amount=("disputed_amount_total", "sum"),
        risky_status_transactions=("merchant_status_risk_flag", "sum")
    )
    .reset_index()
)

# Merchant chargeback ratio
merchant_risk["chargeback_ratio"] = (
    merchant_risk["chargeback_transactions"] /
    merchant_risk["transaction_count"]
)

# Fraud chargeback ratio
merchant_risk["fraud_chargeback_ratio"] = (
    merchant_risk["fraud_chargeback_transactions"] /
    merchant_risk["transaction_count"]
)

# Sort by chargeback count
merchant_risk_by_chargebacks = merchant_risk.sort_values(
    "chargeback_transactions",
    ascending=False
)

print("Unique merchants in transaction fact table:",
      merchant_risk["merchant_id"].nunique())

print("\nTop 10 merchants by chargeback transactions:")

display(
    merchant_risk_by_chargebacks[
        [
            "merchant_id",
            "merchant_name",
            "merchant_category_final",
            "transaction_count",
            "chargeback_transactions",
            "fraud_chargeback_transactions",
            "disputed_amount",
            "chargeback_ratio"
        ]
    ].head(10)
)

Unique merchants in transaction fact table: 8051

Top 10 merchants by chargeback transactions:


,merchant_id,merchant_name,merchant_category_final,transaction_count,chargeback_transactions,fraud_chargeback_transactions,disputed_amount,chargeback_ratio
3819,MCH5278,"Sarraf, Peri and Ratti",MISC_RETAIL,3,3,0,6303.65,1.000000
5084,MCH6678,NaN,<NA>,4,3,2,10586.90,0.750000
2051,MCH3285,"Prasad, Luthra and Suri",RETAIL,5,3,0,19559.50,0.600000
2260,MCH3520,"Kannan, Bail and Sundaram",APPAREL,8,3,0,9195.18,0.375000
1065,MCH2179,CHANDA-SURA,APPAREL,5,3,2,6273.36,0.600000
1352,MCH2497,NaN,<NA>,5,3,0,2488.27,0.600000
3734,MCH5177,NaN,<NA>,5,3,0,2196.54,0.600000
674,MCH1744,"Mani, Tara and Mane",TRAVEL,3,3,2,3298.93,1.000000
2287,MCH3549,Khalsa and Sons,FOOD_SERVICES,3,3,2,5137.31,1.000000
180,MCH1195,Cherian-Rana,TRANSPORT,7,3,1,7414.65,0.428571


In [26]:
# Inspect merchant transaction-volume distribution

merchant_volume_stats = merchant_risk["transaction_count"].describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
)

print("Merchant transaction-volume distribution:")
print(merchant_volume_stats)

print("\nMerchant count by transaction volume:")
print(
    merchant_risk["transaction_count"]
    .value_counts()
    .sort_index()
    .head(20)
)

Merchant transaction-volume distribution:
count    8051.000000
mean        2.484163
std         1.329023
min         1.000000
50%         2.000000
75%         3.000000
90%         4.000000
95%         5.000000
99%         6.000000
max        10.000000
Name: transaction_count, dtype: float64

Merchant count by transaction volume:
transaction_count
1     2148
2     2452
3     1814
4      979
5      436
6      162
7       47
8       10
9        1
10       2
Name: count, dtype: int64


In [27]:
# Create a meaningful merchant-risk ranking
# Restrict the primary ranking to merchants with >= 5 transactions

merchant_risk_meaningful = merchant_risk[
    merchant_risk["transaction_count"] >= 5
].copy()

# Rank by chargeback ratio
merchant_risk_meaningful = merchant_risk_meaningful.sort_values(
    ["chargeback_ratio", "chargeback_transactions", "disputed_amount"],
    ascending=[False, False, False]
)

print(
    "Merchants with at least 5 transactions:",
    len(merchant_risk_meaningful)
)

print("\nTop 15 merchants by chargeback ratio (minimum 5 transactions):")

display(
    merchant_risk_meaningful[
        [
            "merchant_id",
            "merchant_name",
            "merchant_category_final",
            "transaction_count",
            "chargeback_transactions",
            "fraud_chargeback_transactions",
            "disputed_amount",
            "chargeback_ratio",
            "fraud_chargeback_ratio"
        ]
    ].head(15)
)

Merchants with at least 5 transactions: 658

Top 15 merchants by chargeback ratio (minimum 5 transactions):


,merchant_id,merchant_name,merchant_category_final,transaction_count,chargeback_transactions,fraud_chargeback_transactions,disputed_amount,chargeback_ratio,fraud_chargeback_ratio
2051,MCH3285,"Prasad, Luthra and Suri",RETAIL,5,3,0,19559.50,0.600000,0.000000
2321,MCH3587,Sagar-Raval,HOTEL_LODGING,5,3,1,10230.17,0.600000,0.200000
1065,MCH2179,CHANDA-SURA,APPAREL,5,3,2,6273.36,0.600000,0.400000
1352,MCH2497,NaN,<NA>,5,3,0,2488.27,0.600000,0.000000
3734,MCH5177,NaN,<NA>,5,3,0,2196.54,0.600000,0.000000
5346,MCH6973,NaN,<NA>,6,3,1,10427.30,0.500000,0.166667
180,MCH1195,Cherian-Rana,TRANSPORT,7,3,1,7414.65,0.428571,0.142857
1513,MCH2681,Upadhyay-Bava,HOTEL_LODGING,5,2,0,19412.12,0.400000,0.000000
7049,MCH8869,Tella-Roy,MEDICAL_STORE,5,2,0,17812.49,0.400000,0.000000
2011,MCH3232,"Sarraf,ChaudhuriandAcharya",FOOD_SERVICES,5,2,0,14888.36,0.400000,0.000000


In [28]:
# Identify merchants with strong chargeback evidence
# Criteria:
# 1. At least 5 transactions
# 2. At least 2 chargeback transactions

merchant_risk_priority = merchant_risk[
    (merchant_risk["transaction_count"] >= 5) &
    (merchant_risk["chargeback_transactions"] >= 2)
].copy()

merchant_risk_priority = merchant_risk_priority.sort_values(
    ["chargeback_transactions", "chargeback_ratio", "disputed_amount"],
    ascending=[False, False, False]
)

print(
    "Priority merchants (>=5 transactions and >=2 chargebacks):",
    len(merchant_risk_priority)
)

print("\nTop 15 priority merchants:")

display(
    merchant_risk_priority[
        [
            "merchant_id",
            "merchant_name",
            "merchant_category_final",
            "transaction_count",
            "transaction_value",
            "chargeback_transactions",
            "fraud_chargeback_transactions",
            "disputed_amount",
            "chargeback_ratio",
            "fraud_chargeback_ratio"
        ]
    ].head(15)
)

Priority merchants (>=5 transactions and >=2 chargebacks): 88

Top 15 priority merchants:


,merchant_id,merchant_name,merchant_category_final,transaction_count,transaction_value,chargeback_transactions,fraud_chargeback_transactions,disputed_amount,chargeback_ratio,fraud_chargeback_ratio
2051,MCH3285,"Prasad, Luthra and Suri",RETAIL,5,46673.46,3,0,19559.50,0.600000,0.000000
2321,MCH3587,Sagar-Raval,HOTEL_LODGING,5,53437.81,3,1,10230.17,0.600000,0.200000
1065,MCH2179,CHANDA-SURA,APPAREL,5,41623.73,3,2,6273.36,0.600000,0.400000
1352,MCH2497,NaN,<NA>,5,54937.75,3,0,2488.27,0.600000,0.000000
3734,MCH5177,NaN,<NA>,5,80599.72,3,0,2196.54,0.600000,0.000000
5346,MCH6973,NaN,<NA>,6,71718.81,3,1,10427.30,0.500000,0.166667
180,MCH1195,Cherian-Rana,TRANSPORT,7,48300.89,3,1,7414.65,0.428571,0.142857
2260,MCH3520,"Kannan, Bail and Sundaram",APPAREL,8,65366.44,3,0,9195.18,0.375000,0.000000
1513,MCH2681,Upadhyay-Bava,HOTEL_LODGING,5,71746.48,2,0,19412.12,0.400000,0.000000
7049,MCH8869,Tella-Roy,MEDICAL_STORE,5,32095.55,2,0,17812.49,0.400000,0.000000


In [29]:
print("Columns in category_risk_ranked:")
print(category_risk_ranked.columns.tolist())

Columns in category_risk_ranked:
['merchant_category_final', 'transaction_count', 'transaction_value', 'chargeback_transactions', 'fraud_chargeback_transactions', 'disputed_amount', 'chargeback_to_transaction_ratio', 'fraud_chargeback_ratio', 'chargeback_ratio_rank']


In [30]:
# Top merchant categories by chargeback-to-transaction ratio
# Exclude transactions where merchant category is unavailable

category_top10 = category_risk_ranked[
    category_risk_ranked["merchant_category_final"].notna()
].copy()

category_top10 = category_top10.sort_values(
    [
        "chargeback_to_transaction_ratio",
        "chargeback_transactions",
        "disputed_amount"
    ],
    ascending=[False, False, False]
).head(10)

print("Top 10 merchant categories by chargeback-to-transaction ratio:")

display(
    category_top10[
        [
            "merchant_category_final",
            "transaction_count",
            "transaction_value",
            "chargeback_transactions",
            "fraud_chargeback_transactions",
            "disputed_amount",
            "chargeback_to_transaction_ratio",
            "fraud_chargeback_ratio"
        ]
    ]
)

Top 10 merchant categories by chargeback-to-transaction ratio:


,merchant_category_final,transaction_count,transaction_value,chargeback_transactions,fraud_chargeback_transactions,disputed_amount,chargeback_to_transaction_ratio,fraud_chargeback_ratio
10,OTHER,195,2217035.07,35,11,131017.56,0.179487,0.056410
14,STATIONERY,256,2979719.27,36,12,139225.25,0.140625,0.046875
7,MEDICAL_STORE,208,2128353.04,29,11,112065.59,0.139423,0.052885
6,MEDICAL,158,1774900.50,22,8,70702.63,0.139241,0.050633
15,TELECOM,454,4598870.63,63,23,190165.63,0.138767,0.050661
13,RETAIL,289,3040042.12,40,12,121780.98,0.138408,0.041522
12,PHONE SERVICE,240,2708885.66,33,11,112240.85,0.137500,0.045833
16,TRANSPORT,816,8571969.26,111,36,296866.51,0.136029,0.044118
8,MISC_RETAIL,669,6890982.79,85,29,196894.50,0.127055,0.043348
17,TRAVEL,182,1804988.01,23,10,43858.00,0.126374,0.054945


In [31]:
# Rank merchant categories by total chargeback volume

category_chargeback_volume = category_risk_ranked[
    category_risk_ranked["merchant_category_final"].notna()
].copy()

category_chargeback_volume = category_chargeback_volume.sort_values(
    ["chargeback_transactions", "disputed_amount"],
    ascending=[False, False]
).head(10)

print("Top 10 merchant categories by chargeback volume:")

display(
    category_chargeback_volume[
        [
            "merchant_category_final",
            "transaction_count",
            "transaction_value",
            "chargeback_transactions",
            "fraud_chargeback_transactions",
            "disputed_amount",
            "chargeback_to_transaction_ratio",
            "fraud_chargeback_ratio"
        ]
    ]
)

Top 10 merchant categories by chargeback volume:


,merchant_category_final,transaction_count,transaction_value,chargeback_transactions,fraud_chargeback_transactions,disputed_amount,chargeback_to_transaction_ratio,fraud_chargeback_ratio
0,APPAREL,981,10442794.30,120,48,330564.34,0.122324,0.048930
16,TRANSPORT,816,8571969.26,111,36,296866.51,0.136029,0.044118
5,HOTEL_LODGING,906,9224285.94,107,43,341634.95,0.118102,0.047461
3,FOOD_SERVICES,918,9941427.39,105,35,272684.87,0.114379,0.038126
4,GROCERY,855,9607515.80,103,30,340256.20,0.120468,0.035088
8,MISC_RETAIL,669,6890982.79,85,29,196894.50,0.127055,0.043348
1,BOOKS,657,6881434.21,81,24,261967.44,0.123288,0.036530
2,DEPARTMENT STORE,749,8231786.15,77,31,211144.73,0.102804,0.041389
15,TELECOM,454,4598870.63,63,23,190165.63,0.138767,0.050661
11,PHARMACY,496,5462561.08,58,16,176024.88,0.116935,0.032258


In [32]:
# Final category risk analytics table for Power BI

category_risk_final = category_risk_ranked[
    category_risk_ranked["merchant_category_final"].notna()
].copy()

category_risk_final = category_risk_final[
    [
        "merchant_category_final",
        "transaction_count",
        "transaction_value",
        "chargeback_transactions",
        "fraud_chargeback_transactions",
        "disputed_amount",
        "chargeback_to_transaction_ratio",
        "fraud_chargeback_ratio",
        "chargeback_ratio_rank"
    ]
].sort_values(
    "chargeback_ratio_rank"
)

print("Final category risk table:", category_risk_final.shape)

display(category_risk_final)

Final category risk table: (18, 9)


,merchant_category_final,transaction_count,transaction_value,chargeback_transactions,fraud_chargeback_transactions,disputed_amount,chargeback_to_transaction_ratio,fraud_chargeback_ratio,chargeback_ratio_rank
10,OTHER,195,2217035.07,35,11,131017.56,0.179487,0.056410,1
14,STATIONERY,256,2979719.27,36,12,139225.25,0.140625,0.046875,2
7,MEDICAL_STORE,208,2128353.04,29,11,112065.59,0.139423,0.052885,3
6,MEDICAL,158,1774900.50,22,8,70702.63,0.139241,0.050633,4
15,TELECOM,454,4598870.63,63,23,190165.63,0.138767,0.050661,5
13,RETAIL,289,3040042.12,40,12,121780.98,0.138408,0.041522,6
12,PHONE SERVICE,240,2708885.66,33,11,112240.85,0.137500,0.045833,7
16,TRANSPORT,816,8571969.26,111,36,296866.51,0.136029,0.044118,8
8,MISC_RETAIL,669,6890982.79,85,29,196894.50,0.127055,0.043348,9
17,TRAVEL,182,1804988.01,23,10,43858.00,0.126374,0.054945,10


In [33]:
import pandas as pd

# Reload the enriched transaction fact table created in previous steps
fact_transactions_enriched = pd.read_csv(
    "../data/processed/fact_transactions_enriched.csv"
)

print("Fact table shape:", fact_transactions_enriched.shape)
print("Unique transactions:", fact_transactions_enriched["txn_id"].nunique())
print("Duplicate transaction IDs:", fact_transactions_enriched["txn_id"].duplicated().sum())

Fact table shape: (20000, 92)
Unique transactions: 20000
Duplicate transaction IDs: 0


In [34]:
# Inspect existing fraud-risk signal columns

fraud_signal_keywords = [
    "fraud",
    "risk",
    "anomaly",
    "chargeback",
    "velocity",
    "high_value",
    "kyc_rejected",
    "merchant_status"
]

fraud_signal_columns = [
    col for col in fact_transactions_enriched.columns
    if any(keyword in col.lower() for keyword in fraud_signal_keywords)
]

print("Fraud/risk-related columns:")
for col in fraud_signal_columns:
    print("-", col)

print("\nTotal fraud/risk-related columns:", len(fraud_signal_columns))

Fraud/risk-related columns:
- risk_segment
- merchant_status
- chargeback_count
- fraud_unauthorized_chargeback_count
- max_chargeback_severity
- chargeback_flag
- fraud_chargeback_flag
- high_value_transaction_flag
- high_risk_user_flag
- kyc_rejected_transaction_flag
- merchant_status_risk_flag
- high_user_velocity_flag
- high_merchant_velocity_flag
- velocity_anomaly_flag
- user_chargeback_count
- repeated_chargeback_user_flag
- merchant_chargeback_count
- high_chargeback_merchant_flag
- strong_identity_anomaly_flag_x
- strong_identity_anomaly_flag_y
- strong_identity_anomaly_flag

Total fraud/risk-related columns: 21


In [35]:
# Check the distribution of the main fraud-risk signals

risk_signal_columns = [
    "chargeback_flag",
    "fraud_chargeback_flag",
    "high_value_transaction_flag",
    "high_risk_user_flag",
    "kyc_rejected_transaction_flag",
    "merchant_status_risk_flag",
    "velocity_anomaly_flag",
    "repeated_chargeback_user_flag",
    "high_chargeback_merchant_flag",
    "strong_identity_anomaly_flag"
]

risk_signal_summary = pd.DataFrame({
    "signal": risk_signal_columns,
    "flagged_transactions": [
        fact_transactions_enriched[col].fillna(0).astype(int).sum()
        for col in risk_signal_columns
    ]
})

risk_signal_summary["flagged_percentage"] = (
    risk_signal_summary["flagged_transactions"]
    / len(fact_transactions_enriched)
    * 100
)

display(risk_signal_summary)

,signal,flagged_transactions,flagged_percentage
0,chargeback_flag,2451,12.255
1,fraud_chargeback_flag,873,4.365
2,high_value_transaction_flag,879,4.395
3,high_risk_user_flag,686,3.430
4,kyc_rejected_transaction_flag,491,2.455
5,merchant_status_risk_flag,738,3.690
6,velocity_anomaly_flag,460,2.300
7,repeated_chargeback_user_flag,266,1.330
8,high_chargeback_merchant_flag,242,1.210
9,strong_identity_anomaly_flag,1141,5.705


In [36]:
# Create an explainable 100-point fraud-risk score

risk_weights = {
    "fraud_chargeback_flag": 30,
    "chargeback_flag": 15,
    "strong_identity_anomaly_flag": 15,
    "repeated_chargeback_user_flag": 10,
    "high_chargeback_merchant_flag": 10,
    "high_risk_user_flag": 5,
    "kyc_rejected_transaction_flag": 5,
    "velocity_anomaly_flag": 5,
    "high_value_transaction_flag": 3,
    "merchant_status_risk_flag": 2
}

# Calculate weighted contribution from each signal
for signal, weight in risk_weights.items():
    fact_transactions_enriched[f"{signal}_points"] = (
        fact_transactions_enriched[signal]
        .fillna(0)
        .astype(int)
        * weight
    )

# Total explainable risk score
risk_point_columns = [
    f"{signal}_points"
    for signal in risk_weights
]

fact_transactions_enriched["fraud_risk_score"] = (
    fact_transactions_enriched[risk_point_columns]
    .sum(axis=1)
)

print("Risk score created successfully.")

print(
    "\nScore range:",
    fact_transactions_enriched["fraud_risk_score"].min(),
    "to",
    fact_transactions_enriched["fraud_risk_score"].max()
)

print(
    "Maximum possible score:",
    sum(risk_weights.values())
)

display(
    fact_transactions_enriched[
        ["txn_id", "fraud_risk_score"] + risk_point_columns
    ].head(10)
)

Risk score created successfully.

Score range: 0 to 85
Maximum possible score: 100


,txn_id,fraud_risk_score,fraud_chargeback_flag_points,chargeback_flag_points,strong_identity_anomaly_flag_points,repeated_chargeback_user_flag_points,high_chargeback_merchant_flag_points,high_risk_user_flag_points,kyc_rejected_transaction_flag_points,velocity_anomaly_flag_points,high_value_transaction_flag_points,merchant_status_risk_flag_points
0,TXN00011869,15,0,15,0,0,0,0,0,0,0,0
1,TXN00010383,30,0,15,15,0,0,0,0,0,0,0
2,TXN00008297,0,0,0,0,0,0,0,0,0,0,0
3,TXN00006448,0,0,0,0,0,0,0,0,0,0,0
4,TXN00018792,0,0,0,0,0,0,0,0,0,0,0
5,TXN00000400,5,0,0,0,0,0,0,5,0,0,0
6,TXN00015249,0,0,0,0,0,0,0,0,0,0,0
7,TXN00007121,0,0,0,0,0,0,0,0,0,0,0
8,TXN00008460,0,0,0,0,0,0,0,0,0,0,0
9,TXN00002541,0,0,0,0,0,0,0,0,0,0,0


In [37]:
# Convert the numerical fraud-risk score into explainable risk bands

fact_transactions_enriched["fraud_risk_band"] = pd.cut(
    fact_transactions_enriched["fraud_risk_score"],
    bins=[-1, 19, 39, 59, 100],
    labels=["LOW", "MEDIUM", "HIGH", "CRITICAL"]
)

risk_band_summary = (
    fact_transactions_enriched["fraud_risk_band"]
    .value_counts()
    .sort_index()
    .reset_index()
)

risk_band_summary.columns = [
    "risk_band",
    "transaction_count"
]

risk_band_summary["percentage"] = (
    risk_band_summary["transaction_count"]
    / len(fact_transactions_enriched)
    * 100
)

print("Fraud-risk band distribution:")

display(risk_band_summary)

Fraud-risk band distribution:


,risk_band,transaction_count,percentage
0,LOW,18553,92.765
1,MEDIUM,565,2.825
2,HIGH,788,3.940
3,CRITICAL,94,0.470


In [38]:
# Check the exact merchant/category columns currently available

merchant_category_columns = [
    col for col in fact_transactions_enriched.columns
    if "merchant" in col.lower() or "category" in col.lower()
]

print("Merchant/category columns:")
for col in merchant_category_columns:
    print("-", col)

Merchant/category columns:
- merchant_id
- merchant_match_flag
- merchant_name
- mcc_merchant
- merchant_category
- city_merchant
- state_merchant
- merchant_status
- merchant_id_missing_flag
- merchant_status_risk_flag
- merchant_daily_txn_count
- high_merchant_velocity_flag
- merchant_chargeback_count
- high_chargeback_merchant_flag
- high_chargeback_merchant_flag_points
- merchant_status_risk_flag_points


In [39]:
# Identify the highest-risk transactions

top_risk_transactions = fact_transactions_enriched.sort_values(
    ["fraud_risk_score", "chargeback_flag", "fraud_chargeback_flag"],
    ascending=[False, False, False]
).copy()

print("Top 20 highest-risk transactions:")

display(
    top_risk_transactions[
        [
            "txn_id",
            "user_id",
            "merchant_id",
            "merchant_name",
            "merchant_category",
            "amount_numeric",
            "status_clean",
            "fraud_risk_score",
            "fraud_risk_band",
            "chargeback_flag",
            "fraud_chargeback_flag",
            "strong_identity_anomaly_flag",
            "repeated_chargeback_user_flag",
            "high_chargeback_merchant_flag",
            "high_risk_user_flag",
            "kyc_rejected_transaction_flag",
            "velocity_anomaly_flag",
            "high_value_transaction_flag",
            "merchant_status_risk_flag"
        ]
    ].head(20)
)

Top 20 highest-risk transactions:


,txn_id,user_id,merchant_id,merchant_name,merchant_category,amount_numeric,status_clean,fraud_risk_score,fraud_risk_band,chargeback_flag,fraud_chargeback_flag,strong_identity_anomaly_flag,repeated_chargeback_user_flag,high_chargeback_merchant_flag,high_risk_user_flag,kyc_rejected_transaction_flag,velocity_anomaly_flag,high_value_transaction_flag,merchant_status_risk_flag
13953,TXN00001975,USR99484,MCH5922,NaN,NaN,7475.15,SUCCESS,85,CRITICAL,True,True,1.0,1,1,False,False,1,False,False
15102,TXN00011255,USR95217,MCH9572,NaN,NaN,8219.44,SUCCESS,85,CRITICAL,True,True,1.0,1,1,False,True,0,False,False
14294,TXN00017748,USR61630,MCH9572,NaN,NaN,6599.82,SUCCESS,80,CRITICAL,True,True,1.0,1,1,False,False,0,False,False
4150,TXN00015779,USR37017,MCH4560,Shankar Ltd,Telecom,839.58,SUCCESS,75,CRITICAL,True,True,1.0,1,0,True,False,0,False,False
14397,TXN00004673,USR93634,MCH5953,"Sandhu, Srinivas and Thakur",garments,18137.04,SUCCESS,75,CRITICAL,True,True,1.0,1,0,False,True,0,False,False
4546,TXN00013202,USR52672,MCH2179,CHANDA-SURA,Fashion,12732.75,SUCCESS,70,CRITICAL,True,True,1.0,0,1,False,False,0,False,False
16467,TXN00019786,USR59386,MCH6257,"Parsa, Rana and Raja",FOOD_SERVICES,14465.41,SUCCESS,70,CRITICAL,True,True,1.0,0,1,False,False,0,False,False
19091,TXN00019212,USR60294,MCH6994,NaN,NaN,18650.46,SUCCESS,70,CRITICAL,True,True,1.0,1,0,False,False,0,False,False
6765,TXN00006912,USR52813,MCH5057,NaN,NaN,24446.56,FAILED,68,CRITICAL,True,True,0.0,1,1,False,False,0,True,False
8358,TXN00019245,USR52299,MCH6960,DAS-DHAWAN,FOOD_SERVICES,24866.41,SUCCESS,68,CRITICAL,True,True,0.0,1,1,False,False,0,True,False


In [40]:
# Analyze which risk signals contribute most to HIGH/CRITICAL transactions

high_critical = fact_transactions_enriched[
    fact_transactions_enriched["fraud_risk_band"].isin(["HIGH", "CRITICAL"])
].copy()

risk_contribution_summary = pd.DataFrame({
    "risk_signal": list(risk_weights.keys()),
    "transactions_flagged": [
        high_critical[col].fillna(0).astype(int).sum()
        for col in risk_weights.keys()
    ],
    "points_per_signal": list(risk_weights.values())
})

risk_contribution_summary["percentage_of_high_critical"] = (
    risk_contribution_summary["transactions_flagged"]
    / len(high_critical)
    * 100
)

risk_contribution_summary = risk_contribution_summary.sort_values(
    "transactions_flagged",
    ascending=False
)

print(
    "HIGH + CRITICAL transactions:",
    len(high_critical)
)

display(risk_contribution_summary)

HIGH + CRITICAL transactions: 882


,risk_signal,transactions_flagged,points_per_signal,percentage_of_high_critical
1,chargeback_flag,882,15,100.000000
0,fraud_chargeback_flag,873,30,98.979592
3,repeated_chargeback_user_flag,115,10,13.038549
2,strong_identity_anomaly_flag,67,15,7.596372
4,high_chargeback_merchant_flag,59,10,6.689342
8,high_value_transaction_flag,42,3,4.761905
5,high_risk_user_flag,33,5,3.741497
9,merchant_status_risk_flag,33,2,3.741497
7,velocity_anomaly_flag,25,5,2.834467
6,kyc_rejected_transaction_flag,21,5,2.380952


In [41]:
# Create an explainable list of reasons for each transaction's risk score

def get_risk_reasons(row):
    reasons = []

    if row["fraud_chargeback_flag"] == 1:
        reasons.append("Fraud/unauthorized chargeback")

    if row["chargeback_flag"] == 1:
        reasons.append("Chargeback")

    if row["strong_identity_anomaly_flag"] == 1:
        reasons.append("Strong identity anomaly")

    if row["repeated_chargeback_user_flag"] == 1:
        reasons.append("Repeated user chargebacks")

    if row["high_chargeback_merchant_flag"] == 1:
        reasons.append("High-chargeback merchant")

    if row["high_risk_user_flag"] == 1:
        reasons.append("High-risk user")

    if row["kyc_rejected_transaction_flag"] == 1:
        reasons.append("KYC rejected/failed")

    if row["velocity_anomaly_flag"] == 1:
        reasons.append("High transaction velocity")

    if row["high_value_transaction_flag"] == 1:
        reasons.append("High-value transaction")

    if row["merchant_status_risk_flag"] == 1:
        reasons.append("Risky merchant status")

    if not reasons:
        reasons.append("No major risk signals")

    return "; ".join(reasons)


fact_transactions_enriched["risk_reasons"] = (
    fact_transactions_enriched.apply(get_risk_reasons, axis=1)
)

print("Risk-reason column created successfully.")

display(
    fact_transactions_enriched[
        [
            "txn_id",
            "fraud_risk_score",
            "fraud_risk_band",
            "risk_reasons"
        ]
    ]
    .sort_values("fraud_risk_score", ascending=False)
    .head(15)
)

Risk-reason column created successfully.


,txn_id,fraud_risk_score,fraud_risk_band,risk_reasons
15102,TXN00011255,85,CRITICAL,Fraud/unauthorized chargeback; Chargeback; Str...
13953,TXN00001975,85,CRITICAL,Fraud/unauthorized chargeback; Chargeback; Str...
14294,TXN00017748,80,CRITICAL,Fraud/unauthorized chargeback; Chargeback; Str...
4150,TXN00015779,75,CRITICAL,Fraud/unauthorized chargeback; Chargeback; Str...
14397,TXN00004673,75,CRITICAL,Fraud/unauthorized chargeback; Chargeback; Str...
4546,TXN00013202,70,CRITICAL,Fraud/unauthorized chargeback; Chargeback; Str...
16467,TXN00019786,70,CRITICAL,Fraud/unauthorized chargeback; Chargeback; Str...
19091,TXN00019212,70,CRITICAL,Fraud/unauthorized chargeback; Chargeback; Str...
11734,TXN00009979,68,CRITICAL,Fraud/unauthorized chargeback; Chargeback; Rep...
6765,TXN00006912,68,CRITICAL,Fraud/unauthorized chargeback; Chargeback; Rep...


In [42]:
# Create the investigation population
# Focus on HIGH and CRITICAL risk transactions

investigation_cases = fact_transactions_enriched[
    fact_transactions_enriched["fraud_risk_band"].isin(["HIGH", "CRITICAL"])
].copy()

# Sort highest-risk cases first
investigation_cases = investigation_cases.sort_values(
    ["fraud_risk_score", "fraud_chargeback_flag", "chargeback_flag"],
    ascending=[False, False, False]
)

print("Investigation cases:", len(investigation_cases))
print(
    "Unique transaction IDs:",
    investigation_cases["txn_id"].nunique()
)

print("\nRisk-band distribution:")
display(
    investigation_cases["fraud_risk_band"]
    .value_counts()
    .sort_index()
    .rename_axis("risk_band")
    .reset_index(name="transaction_count")
)

Investigation cases: 882
Unique transaction IDs: 882

Risk-band distribution:


,risk_band,transaction_count
0,LOW,0
1,MEDIUM,0
2,HIGH,788
3,CRITICAL,94


In [43]:
# Assign investigation priority

def assign_investigation_priority(row):
    if row["fraud_risk_band"] == "CRITICAL":
        return "P1 - Immediate Review"
    
    if row["fraud_chargeback_flag"] == 1:
        return "P2 - High Priority"
    
    return "P3 - Review"

investigation_cases["investigation_priority"] = (
    investigation_cases.apply(
        assign_investigation_priority,
        axis=1
    )
)

priority_summary = (
    investigation_cases["investigation_priority"]
    .value_counts()
    .rename_axis("investigation_priority")
    .reset_index(name="transaction_count")
)

print("Investigation priority distribution:")

display(priority_summary)

Investigation priority distribution:


,investigation_priority,transaction_count
0,P2 - High Priority,779
1,P1 - Immediate Review,94
2,P3 - Review,9


In [44]:
# Build the investigation evidence table

investigation_evidence = investigation_cases[
    [
        "txn_id",
        "timestamp",
        "user_id",
        "merchant_id",
        "merchant_name",
        "merchant_category",
        "amount_numeric",
        "status_clean",
        "fraud_risk_score",
        "fraud_risk_band",
        "investigation_priority",
        "risk_reasons",
        "chargeback_count",
        "fraud_unauthorized_chargeback_count",
        "max_chargeback_severity",
        "user_chargeback_count",
        "merchant_chargeback_count",
        "strong_identity_anomaly_flag",
        "repeated_chargeback_user_flag",
        "high_chargeback_merchant_flag",
        "high_risk_user_flag",
        "kyc_rejected_transaction_flag",
        "velocity_anomaly_flag",
        "high_value_transaction_flag",
        "merchant_status_risk_flag",
        "kyc_status",
        "risk_segment",
        "merchant_status",
        "kyc_match_flag",
        "merchant_match_flag"
    ]
].copy()

print("Investigation evidence table shape:", investigation_evidence.shape)

display(
    investigation_evidence.head(10)
)

Investigation evidence table shape: (882, 30)


,txn_id,timestamp,user_id,merchant_id,merchant_name,merchant_category,amount_numeric,status_clean,fraud_risk_score,fraud_risk_band,...,high_risk_user_flag,kyc_rejected_transaction_flag,velocity_anomaly_flag,high_value_transaction_flag,merchant_status_risk_flag,kyc_status,risk_segment,merchant_status,kyc_match_flag,merchant_match_flag
13953,TXN00001975,2026-02-01 13:44:13,USR99484,MCH5922,NaN,NaN,7475.15,SUCCESS,85,CRITICAL,...,False,False,1,False,False,APPROVED,MEDIUM,NaN,True,False
15102,TXN00011255,2026-03-08 00:00:00,USR95217,MCH9572,NaN,NaN,8219.44,SUCCESS,85,CRITICAL,...,False,True,0,False,False,REJECTED,LOW,NaN,True,False
14294,TXN00017748,2026-02-02 09:40:09,USR61630,MCH9572,NaN,NaN,6599.82,SUCCESS,80,CRITICAL,...,False,False,0,False,False,VERIFIED,LOW,NaN,True,False
4150,TXN00015779,2026-01-09 05:28:02,USR37017,MCH4560,Shankar Ltd,Telecom,839.58,SUCCESS,75,CRITICAL,...,True,False,0,False,False,VERIFIED,HIGH,Active,True,True
14397,TXN00004673,2026-01-22 03:34:49,USR93634,MCH5953,"Sandhu, Srinivas and Thakur",garments,18137.04,SUCCESS,75,CRITICAL,...,False,True,0,False,False,FAILED,LOW,Live,True,True
4546,TXN00013202,2026-01-02 20:12:41,USR52672,MCH2179,CHANDA-SURA,Fashion,12732.75,SUCCESS,70,CRITICAL,...,False,False,0,False,False,VERIFIED,LOW,ACTIVE,True,True
16467,TXN00019786,2026-01-17 02:40:53,USR59386,MCH6257,"Parsa, Rana and Raja",FOOD_SERVICES,14465.41,SUCCESS,70,CRITICAL,...,False,False,0,False,False,VERIFIED,LOW,A,True,True
19091,TXN00019212,2026-01-05 16:31:08,USR60294,MCH6994,NaN,NaN,18650.46,SUCCESS,70,CRITICAL,...,False,False,0,False,False,VERIFIED,MEDIUM,NaN,True,False
6765,TXN00006912,2026-02-19 22:25:37,USR52813,MCH5057,NaN,NaN,24446.56,FAILED,68,CRITICAL,...,False,False,0,True,False,VERIFIED,MEDIUM,NaN,True,False
8358,TXN00019245,2026-01-12 12:14:38,USR52299,MCH6960,DAS-DHAWAN,FOOD_SERVICES,24866.41,SUCCESS,68,CRITICAL,...,False,False,0,True,False,APPROVED,LOW,Active,True,True


In [45]:
# Add recommended investigation actions

def investigation_action(row):
    if row["investigation_priority"] == "P1 - Immediate Review":
        if row["fraud_unauthorized_chargeback_count"] > 0:
            return "Immediately review fraud/unauthorized chargeback evidence"
        return "Immediately review combined critical-risk signals"

    if row["investigation_priority"] == "P2 - High Priority":
        if row["fraud_unauthorized_chargeback_count"] > 0:
            return "Review fraud/unauthorized chargeback evidence"
        return "Review supporting risk signals"

    return "Secondary review"


investigation_evidence["recommended_action"] = (
    investigation_evidence.apply(
        investigation_action,
        axis=1
    )
)

print("Recommended investigation actions added.")

display(
    investigation_evidence[
        [
            "txn_id",
            "fraud_risk_score",
            "fraud_risk_band",
            "investigation_priority",
            "risk_reasons",
            "recommended_action"
        ]
    ].head(15)
)

Recommended investigation actions added.


,txn_id,fraud_risk_score,fraud_risk_band,investigation_priority,risk_reasons,recommended_action
13953,TXN00001975,85,CRITICAL,P1 - Immediate Review,Fraud/unauthorized chargeback; Chargeback; Str...,Immediately review fraud/unauthorized chargeba...
15102,TXN00011255,85,CRITICAL,P1 - Immediate Review,Fraud/unauthorized chargeback; Chargeback; Str...,Immediately review fraud/unauthorized chargeba...
14294,TXN00017748,80,CRITICAL,P1 - Immediate Review,Fraud/unauthorized chargeback; Chargeback; Str...,Immediately review fraud/unauthorized chargeba...
4150,TXN00015779,75,CRITICAL,P1 - Immediate Review,Fraud/unauthorized chargeback; Chargeback; Str...,Immediately review fraud/unauthorized chargeba...
14397,TXN00004673,75,CRITICAL,P1 - Immediate Review,Fraud/unauthorized chargeback; Chargeback; Str...,Immediately review fraud/unauthorized chargeba...
4546,TXN00013202,70,CRITICAL,P1 - Immediate Review,Fraud/unauthorized chargeback; Chargeback; Str...,Immediately review fraud/unauthorized chargeba...
16467,TXN00019786,70,CRITICAL,P1 - Immediate Review,Fraud/unauthorized chargeback; Chargeback; Str...,Immediately review fraud/unauthorized chargeba...
19091,TXN00019212,70,CRITICAL,P1 - Immediate Review,Fraud/unauthorized chargeback; Chargeback; Str...,Immediately review fraud/unauthorized chargeba...
6765,TXN00006912,68,CRITICAL,P1 - Immediate Review,Fraud/unauthorized chargeback; Chargeback; Rep...,Immediately review fraud/unauthorized chargeba...
8358,TXN00019245,68,CRITICAL,P1 - Immediate Review,Fraud/unauthorized chargeback; Chargeback; Rep...,Immediately review fraud/unauthorized chargeba...


In [46]:
# Inspect the exact columns available in the investigation evidence table

print("Investigation evidence columns:")
for col in investigation_evidence.columns:
    print("-", col)

Investigation evidence columns:
- txn_id
- timestamp
- user_id
- merchant_id
- merchant_name
- merchant_category
- amount_numeric
- status_clean
- fraud_risk_score
- fraud_risk_band
- investigation_priority
- risk_reasons
- chargeback_count
- fraud_unauthorized_chargeback_count
- max_chargeback_severity
- user_chargeback_count
- merchant_chargeback_count
- strong_identity_anomaly_flag
- repeated_chargeback_user_flag
- high_chargeback_merchant_flag
- high_risk_user_flag
- kyc_rejected_transaction_flag
- velocity_anomaly_flag
- high_value_transaction_flag
- merchant_status_risk_flag
- kyc_status
- risk_segment
- merchant_status
- kyc_match_flag
- merchant_match_flag
- recommended_action


In [47]:
# Summarize the investigation queue by priority

investigation_priority_summary = (
    investigation_evidence
    .groupby("investigation_priority", observed=True)
    .agg(
        transaction_count=("txn_id", "count"),
        total_transaction_value=("amount_numeric", "sum"),
        fraud_chargeback_cases=(
            "fraud_unauthorized_chargeback_count",
            lambda x: (x > 0).sum()
        ),
        identity_anomaly_cases=(
            "strong_identity_anomaly_flag",
            "sum"
        ),
        repeated_chargeback_user_cases=(
            "repeated_chargeback_user_flag",
            "sum"
        ),
        high_chargeback_merchant_cases=(
            "high_chargeback_merchant_flag",
            "sum"
        ),
        high_risk_user_cases=(
            "high_risk_user_flag",
            "sum"
        )
    )
    .reset_index()
)

print("Investigation queue summary:")

display(
    investigation_priority_summary
    .sort_values("investigation_priority")
)

Investigation queue summary:


,investigation_priority,transaction_count,total_transaction_value,fraud_chargeback_cases,identity_anomaly_cases,repeated_chargeback_user_cases,high_chargeback_merchant_cases,high_risk_user_cases
0,P1 - Immediate Review,94,994151.62,94,60.0,39,30,9
1,P2 - High Priority,779,8470152.52,779,0.0,68,24,23
2,P3 - Review,9,134370.12,0,7.0,8,5,1


In [48]:
# Top 20 investigation cases for analyst review

top_investigation_cases = (
    investigation_evidence[
        [
            "txn_id",
            "timestamp",
            "user_id",
            "merchant_id",
            "merchant_name",
            "merchant_category",
            "amount_numeric",
            "status_clean",
            "fraud_risk_score",
            "fraud_risk_band",
            "investigation_priority",
            "risk_reasons",
            "recommended_action"
        ]
    ]
    .sort_values(
        ["fraud_risk_score", "amount_numeric"],
        ascending=[False, False]
    )
    .head(20)
    .reset_index(drop=True)
)

print("Top 20 investigation cases:")

display(top_investigation_cases)

Top 20 investigation cases:


,txn_id,timestamp,user_id,merchant_id,merchant_name,merchant_category,amount_numeric,status_clean,fraud_risk_score,fraud_risk_band,investigation_priority,risk_reasons,recommended_action
0,TXN00011255,2026-03-08 00:00:00,USR95217,MCH9572,NaN,NaN,8219.44,SUCCESS,85,CRITICAL,P1 - Immediate Review,Fraud/unauthorized chargeback; Chargeback; Str...,Immediately review fraud/unauthorized chargeba...
1,TXN00001975,2026-02-01 13:44:13,USR99484,MCH5922,NaN,NaN,7475.15,SUCCESS,85,CRITICAL,P1 - Immediate Review,Fraud/unauthorized chargeback; Chargeback; Str...,Immediately review fraud/unauthorized chargeba...
2,TXN00017748,2026-02-02 09:40:09,USR61630,MCH9572,NaN,NaN,6599.82,SUCCESS,80,CRITICAL,P1 - Immediate Review,Fraud/unauthorized chargeback; Chargeback; Str...,Immediately review fraud/unauthorized chargeba...
3,TXN00004673,2026-01-22 03:34:49,USR93634,MCH5953,"Sandhu, Srinivas and Thakur",garments,18137.04,SUCCESS,75,CRITICAL,P1 - Immediate Review,Fraud/unauthorized chargeback; Chargeback; Str...,Immediately review fraud/unauthorized chargeba...
4,TXN00015779,2026-01-09 05:28:02,USR37017,MCH4560,Shankar Ltd,Telecom,839.58,SUCCESS,75,CRITICAL,P1 - Immediate Review,Fraud/unauthorized chargeback; Chargeback; Str...,Immediately review fraud/unauthorized chargeba...
5,TXN00019212,2026-01-05 16:31:08,USR60294,MCH6994,NaN,NaN,18650.46,SUCCESS,70,CRITICAL,P1 - Immediate Review,Fraud/unauthorized chargeback; Chargeback; Str...,Immediately review fraud/unauthorized chargeba...
6,TXN00019786,2026-01-17 02:40:53,USR59386,MCH6257,"Parsa, Rana and Raja",FOOD_SERVICES,14465.41,SUCCESS,70,CRITICAL,P1 - Immediate Review,Fraud/unauthorized chargeback; Chargeback; Str...,Immediately review fraud/unauthorized chargeba...
7,TXN00013202,2026-01-02 20:12:41,USR52672,MCH2179,CHANDA-SURA,Fashion,12732.75,SUCCESS,70,CRITICAL,P1 - Immediate Review,Fraud/unauthorized chargeback; Chargeback; Str...,Immediately review fraud/unauthorized chargeba...
8,TXN00019245,2026-01-12 12:14:38,USR52299,MCH6960,DAS-DHAWAN,FOOD_SERVICES,24866.41,SUCCESS,68,CRITICAL,P1 - Immediate Review,Fraud/unauthorized chargeback; Chargeback; Rep...,Immediately review fraud/unauthorized chargeba...
9,TXN00013934,2026-02-01 05:54:08,USR77709,MCH5422,sawhney llc,Fashion,24508.25,SUCCESS,68,CRITICAL,P1 - Immediate Review,Fraud/unauthorized chargeback; Chargeback; Rep...,Immediately review fraud/unauthorized chargeba...


In [49]:
# Save investigation outputs for Power BI and further analysis

investigation_evidence.to_csv(
    "../data/processed/investigation_evidence.csv",
    index=False
)

investigation_priority_summary.to_csv(
    "../data/processed/investigation_priority_summary.csv",
    index=False
)

top_investigation_cases.to_csv(
    "../data/processed/top_20_investigation_cases.csv",
    index=False
)

print("Investigation outputs saved successfully.")
print()
print("investigation_evidence:", investigation_evidence.shape)
print("investigation_priority_summary:", investigation_priority_summary.shape)
print("top_20_investigation_cases:", top_investigation_cases.shape)

Investigation outputs saved successfully.

investigation_evidence: (882, 31)
investigation_priority_summary: (3, 8)
top_20_investigation_cases: (20, 13)


In [50]:
# Step 34: Prepare data for fraud-ring / suspicious network analysis

network_columns = [
    "txn_id",
    "user_id",
    "merchant_id",
    "amount_numeric",
    "fraud_risk_score",
    "fraud_risk_band",
    "fraud_chargeback_flag",
    "chargeback_flag",
    "strong_identity_anomaly_flag",
    "repeated_chargeback_user_flag",
    "high_chargeback_merchant_flag"
]

print("Checking required network-analysis columns:")

for col in network_columns:
    print(f"{col}: {'FOUND' if col in fact_transactions_enriched.columns else 'MISSING'}")

Checking required network-analysis columns:
txn_id: FOUND
user_id: FOUND
merchant_id: FOUND
amount_numeric: FOUND
fraud_risk_score: FOUND
fraud_risk_band: FOUND
fraud_chargeback_flag: FOUND
chargeback_flag: FOUND
strong_identity_anomaly_flag: FOUND
repeated_chargeback_user_flag: FOUND
high_chargeback_merchant_flag: FOUND


In [51]:
# Create User -> Merchant network edges from transaction data

network_edges = (
    fact_transactions_enriched[
        [
            "txn_id",
            "user_id",
            "merchant_id",
            "amount_numeric",
            "fraud_risk_score",
            "fraud_risk_band",
            "fraud_chargeback_flag",
            "chargeback_flag",
            "strong_identity_anomaly_flag",
            "repeated_chargeback_user_flag",
            "high_chargeback_merchant_flag"
        ]
    ]
    .copy()
)

# Remove rows where either endpoint is unavailable
network_edges = network_edges[
    network_edges["user_id"].notna() &
    network_edges["merchant_id"].notna()
].copy()

print("Network edge table created.")
print("Rows:", len(network_edges))
print("Unique users:", network_edges["user_id"].nunique())
print("Unique merchants:", network_edges["merchant_id"].nunique())
print("Unique transactions:", network_edges["txn_id"].nunique())

display(network_edges.head())

Network edge table created.
Rows: 20000
Unique users: 17878
Unique merchants: 8051
Unique transactions: 20000


,txn_id,user_id,merchant_id,amount_numeric,fraud_risk_score,fraud_risk_band,fraud_chargeback_flag,chargeback_flag,strong_identity_anomaly_flag,repeated_chargeback_user_flag,high_chargeback_merchant_flag
0,TXN00011869,USR45826,MCH7045,15722.34,15,LOW,False,True,0.0,0,0
1,TXN00010383,USR79397,MCH5031,6362.90,30,MEDIUM,False,True,1.0,0,0
2,TXN00008297,USR87810,MCH9809,15446.19,0,LOW,False,False,0.0,0,0
3,TXN00006448,USR54287,MCH6928,12110.49,0,LOW,False,False,0.0,0,0
4,TXN00018792,USR53865,MCH8121,19432.94,0,LOW,False,False,0.0,0,0


In [52]:
%pip install networkx

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Users\VIKASH\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [55]:
# Reload the final enriched transaction fact table

import pandas as pd

fact_transactions_enriched = pd.read_csv(
    "../data/processed/fact_transactions_enriched.csv"
)

print("Fact transaction table reloaded successfully.")
print("Shape:", fact_transactions_enriched.shape)
print("Unique transactions:", fact_transactions_enriched["txn_id"].nunique())
print(
    "Duplicate transaction IDs:",
    fact_transactions_enriched["txn_id"].duplicated().sum()
)

print("\nRequired network columns:")

required_network_columns = [
    "user_id",
    "merchant_id",
    "amount_numeric",
    "fraud_risk_score",
    "fraud_risk_band",
    "fraud_chargeback_flag",
    "chargeback_flag",
    "strong_identity_anomaly_flag",
    "repeated_chargeback_user_flag",
    "high_chargeback_merchant_flag"
]

for col in required_network_columns:
    print(
        f"{col}: "
        f"{'FOUND' if col in fact_transactions_enriched.columns else 'MISSING'}"
    )

Fact transaction table reloaded successfully.
Shape: (20000, 92)
Unique transactions: 20000
Duplicate transaction IDs: 0

Required network columns:
user_id: FOUND
merchant_id: FOUND
amount_numeric: FOUND
fraud_risk_score: MISSING
fraud_risk_band: MISSING
fraud_chargeback_flag: FOUND
chargeback_flag: FOUND
strong_identity_anomaly_flag: FOUND
repeated_chargeback_user_flag: FOUND
high_chargeback_merchant_flag: FOUND


In [56]:
# RECOVERY: Load the completed fact table from disk

import pandas as pd

fact_transactions_enriched = pd.read_csv(
    "../data/processed/fact_transactions_enriched.csv"
)

print("Fact table loaded successfully.")
print("Shape:", fact_transactions_enriched.shape)
print("Unique transactions:", fact_transactions_enriched["txn_id"].nunique())
print("Duplicate transaction IDs:", fact_transactions_enriched["txn_id"].duplicated().sum())

Fact table loaded successfully.
Shape: (20000, 92)
Unique transactions: 20000
Duplicate transaction IDs: 0


In [58]:
# Check which Step 32 risk-signal columns are available

risk_signal_columns = [
    "fraud_chargeback_flag",
    "chargeback_flag",
    "strong_identity_anomaly_flag",
    "repeated_chargeback_user_flag",
    "high_chargeback_merchant_flag",
    "high_risk_user_flag",
    "kyc_rejected_transaction_flag",
    "velocity_anomaly_flag",
    "high_value_transaction_flag",
    "merchant_status_risk_flag"
]

print("Risk signal availability:")

for col in risk_signal_columns:
    print(
        f"{col}: "
        f"{'FOUND' if col in fact_transactions_enriched.columns else 'MISSING'}"
    )

Risk signal availability:
fraud_chargeback_flag: FOUND
chargeback_flag: FOUND
strong_identity_anomaly_flag: FOUND
repeated_chargeback_user_flag: FOUND
high_chargeback_merchant_flag: FOUND
high_risk_user_flag: FOUND
kyc_rejected_transaction_flag: FOUND
velocity_anomaly_flag: FOUND
high_value_transaction_flag: FOUND
merchant_status_risk_flag: FOUND


In [60]:
# Restore the Step 32 explainable fraud-risk score and risk band

risk_weights = {
    "fraud_chargeback_flag": 30,
    "chargeback_flag": 15,
    "strong_identity_anomaly_flag": 15,
    "repeated_chargeback_user_flag": 10,
    "high_chargeback_merchant_flag": 10,
    "high_risk_user_flag": 5,
    "kyc_rejected_transaction_flag": 5,
    "velocity_anomaly_flag": 5,
    "high_value_transaction_flag": 3,
    "merchant_status_risk_flag": 2
}

# Ensure risk signals are numeric and missing values become 0
for col in risk_weights:
    fact_transactions_enriched[col] = pd.to_numeric(
        fact_transactions_enriched[col],
        errors="coerce"
    ).fillna(0)

# Calculate explainable risk score
fact_transactions_enriched["fraud_risk_score"] = 0

for col, weight in risk_weights.items():
    fact_transactions_enriched["fraud_risk_score"] += (
        fact_transactions_enriched[col] * weight
    )

# Assign risk bands
fact_transactions_enriched["fraud_risk_band"] = pd.cut(
    fact_transactions_enriched["fraud_risk_score"],
    bins=[-1, 19, 39, 59, float("inf")],
    labels=["LOW", "MEDIUM", "HIGH", "CRITICAL"]
)

print("Risk score and risk band restored successfully.")

print("\nRisk score range:")
print(
    "Minimum:",
    fact_transactions_enriched["fraud_risk_score"].min()
)
print(
    "Maximum:",
    fact_transactions_enriched["fraud_risk_score"].max()
)

print("\nRisk band distribution:")
display(
    fact_transactions_enriched["fraud_risk_band"]
    .value_counts()
    .sort_index()
)

Risk score and risk band restored successfully.

Risk score range:
Minimum: 0.0
Maximum: 85.0

Risk band distribution:


fraud_risk_band
LOW         18553
MEDIUM        565
HIGH          788
CRITICAL       94
Name: count, dtype: int64

In [61]:
# Recreate the network edge table after the kernel/environment reset

network_edges = (
    fact_transactions_enriched[
        [
            "txn_id",
            "user_id",
            "merchant_id",
            "amount_numeric",
            "fraud_risk_score",
            "fraud_risk_band",
            "fraud_chargeback_flag",
            "chargeback_flag",
            "strong_identity_anomaly_flag",
            "repeated_chargeback_user_flag",
            "high_chargeback_merchant_flag"
        ]
    ]
    .copy()
)

network_edges = network_edges[
    network_edges["user_id"].notna() &
    network_edges["merchant_id"].notna()
].copy()

print("Network edge table recreated successfully.")
print("Rows:", len(network_edges))
print("Unique users:", network_edges["user_id"].nunique())
print("Unique merchants:", network_edges["merchant_id"].nunique())
print("Unique transactions:", network_edges["txn_id"].nunique())

Network edge table recreated successfully.
Rows: 20000
Unique users: 17878
Unique merchants: 8051
Unique transactions: 20000


In [62]:
# Build the User-Merchant transaction network

import networkx as nx

G = nx.Graph()

for _, row in network_edges.iterrows():
    user_node = f"USER_{row['user_id']}"
    merchant_node = f"MERCHANT_{row['merchant_id']}"

    if G.has_edge(user_node, merchant_node):
        # Update existing user-merchant relationship
        G[user_node][merchant_node]["transaction_count"] += 1
        G[user_node][merchant_node]["total_amount"] += row["amount_numeric"]
        G[user_node][merchant_node]["max_risk_score"] = max(
            G[user_node][merchant_node]["max_risk_score"],
            row["fraud_risk_score"]
        )
        G[user_node][merchant_node]["chargeback_count"] += int(
            row["chargeback_flag"]
        )
        G[user_node][merchant_node]["fraud_chargeback_count"] += int(
            row["fraud_chargeback_flag"]
        )

    else:
        # Create new user-merchant relationship
        G.add_edge(
            user_node,
            merchant_node,
            transaction_count=1,
            total_amount=row["amount_numeric"],
            max_risk_score=row["fraud_risk_score"],
            chargeback_count=int(row["chargeback_flag"]),
            fraud_chargeback_count=int(row["fraud_chargeback_flag"])
        )

print("Network graph created successfully.")
print("Total nodes:", G.number_of_nodes())
print("Total edges:", G.number_of_edges())

user_nodes = [
    n for n in G.nodes
    if n.startswith("USER_")
]

merchant_nodes = [
    n for n in G.nodes
    if n.startswith("MERCHANT_")
]

print("User nodes:", len(user_nodes))
print("Merchant nodes:", len(merchant_nodes))

Network graph created successfully.
Total nodes: 25929
Total edges: 20000
User nodes: 17878
Merchant nodes: 8051


In [63]:
# Find connected components in the User-Merchant network

connected_components = list(nx.connected_components(G))

component_sizes = pd.DataFrame({
    "component_id": range(1, len(connected_components) + 1),
    "node_count": [len(component) for component in connected_components]
})

component_sizes["user_count"] = [
    sum(node.startswith("USER_") for node in component)
    for component in connected_components
]

component_sizes["merchant_count"] = [
    sum(node.startswith("MERCHANT_") for node in component)
    for component in connected_components
]

component_sizes = component_sizes.sort_values(
    "node_count",
    ascending=False
).reset_index(drop=True)

print("Connected components identified:", len(connected_components))
print()
print("Largest connected components:")

display(component_sizes.head(20))

Connected components identified: 5929

Largest connected components:


,component_id,node_count,user_count,merchant_count
0,35,65,45,20
1,968,50,36,14
2,323,49,35,14
3,375,48,35,13
4,314,42,31,11
5,560,39,29,10
6,997,38,28,10
7,46,38,26,12
8,480,37,28,9
9,279,37,27,10


In [67]:
# Fast component-level fraud-risk analysis

# Map every node to its connected component
node_to_component = {}

for component_id, component in enumerate(connected_components, start=1):
    for node in component:
        node_to_component[node] = component_id

# Assign each transaction to its component
network_edges["component_id"] = (
    network_edges["user_id"].map(
        lambda x: node_to_component.get(f"USER_{x}")
    )
)

# Aggregate transaction-level risk signals by component
component_risk = (
    network_edges
    .groupby("component_id")
    .agg(
        transaction_count=("txn_id", "count"),
        transaction_value=("amount_numeric", "sum"),
        chargeback_transactions=("chargeback_flag", "sum"),
        fraud_chargeback_transactions=("fraud_chargeback_flag", "sum"),
        max_risk_score=("fraud_risk_score", "max"),
        identity_anomaly_transactions=(
            "strong_identity_anomaly_flag",
            "sum"
        ),
        repeated_chargeback_user_transactions=(
            "repeated_chargeback_user_flag",
            "sum"
        ),
        high_chargeback_merchant_transactions=(
            "high_chargeback_merchant_flag",
            "sum"
        )
    )
    .reset_index()
)

# Add network size information
component_risk = component_risk.merge(
    component_sizes,
    on="component_id",
    how="left"
)

# Calculate chargeback ratio
component_risk["chargeback_ratio"] = (
    component_risk["chargeback_transactions"]
    / component_risk["transaction_count"]
)

# Reorder useful columns
component_risk = component_risk[
    [
        "component_id",
        "node_count",
        "user_count",
        "merchant_count",
        "transaction_count",
        "transaction_value",
        "chargeback_transactions",
        "fraud_chargeback_transactions",
        "chargeback_ratio",
        "max_risk_score",
        "identity_anomaly_transactions",
        "repeated_chargeback_user_transactions",
        "high_chargeback_merchant_transactions"
    ]
]

print("Fast component risk analysis completed.")
print("Components analyzed:", len(component_risk))

display(
    component_risk
    .sort_values(
        [
            "fraud_chargeback_transactions",
            "max_risk_score",
            "node_count"
        ],
        ascending=[False, False, False]
    )
    .head(20)
)

Fast component risk analysis completed.
Components analyzed: 5929


,component_id,node_count,user_count,merchant_count,transaction_count,transaction_value,chargeback_transactions,fraud_chargeback_transactions,chargeback_ratio,max_risk_score,identity_anomaly_transactions,repeated_chargeback_user_transactions,high_chargeback_merchant_transactions
463,464,18,12,6,17,115123.22,7,4,0.411765,55.0,0.0,2,3
924,925,14,11,3,13,171224.76,4,3,0.307692,68.0,2.0,2,3
236,237,36,28,8,35,283657.03,6,3,0.171429,65.0,1.0,2,7
460,461,27,21,6,26,280137.47,3,3,0.115385,65.0,1.0,2,0
1679,1680,12,8,4,11,83294.80,6,3,0.545455,65.0,0.0,3,4
313,314,42,31,11,41,494059.42,4,3,0.097561,55.0,0.0,1,0
249,250,31,24,7,30,204765.58,5,3,0.166667,55.0,0.0,0,7
352,353,13,10,3,12,102371.04,4,3,0.333333,55.0,1.0,0,4
289,290,32,23,9,31,396132.68,5,3,0.161290,48.0,3.0,0,0
636,637,13,10,3,12,109501.23,3,2,0.250000,85.0,1.0,3,6


In [68]:
# Create an explainable suspicious-network score

component_risk["network_risk_score"] = (
    component_risk["fraud_chargeback_transactions"] * 30
    + component_risk["chargeback_transactions"] * 10
    + component_risk["identity_anomaly_transactions"] * 5
    + component_risk["repeated_chargeback_user_transactions"] * 5
    + component_risk["high_chargeback_merchant_transactions"] * 3
    + component_risk["max_risk_score"] * 0.5
)

# Require meaningful network activity
component_risk["network_activity_flag"] = (
    (component_risk["user_count"] >= 2) &
    (component_risk["merchant_count"] >= 1) &
    (component_risk["transaction_count"] >= 2)
).astype(int)

# Rank suspicious clusters
suspicious_networks = (
    component_risk[
        component_risk["network_activity_flag"] == 1
    ]
    .sort_values(
        [
            "network_risk_score",
            "fraud_chargeback_transactions",
            "chargeback_ratio"
        ],
        ascending=[False, False, False]
    )
    .reset_index(drop=True)
)

suspicious_networks["network_rank"] = (
    suspicious_networks.index + 1
)

print("Suspicious network scoring completed.")
print("Networks with meaningful activity:", len(suspicious_networks))

display(
    suspicious_networks[
        [
            "network_rank",
            "component_id",
            "node_count",
            "user_count",
            "merchant_count",
            "transaction_count",
            "transaction_value",
            "chargeback_transactions",
            "fraud_chargeback_transactions",
            "chargeback_ratio",
            "max_risk_score",
            "network_risk_score"
        ]
    ].head(20)
)

Suspicious network scoring completed.
Networks with meaningful activity: 4180


,network_rank,component_id,node_count,user_count,merchant_count,transaction_count,transaction_value,chargeback_transactions,fraud_chargeback_transactions,chargeback_ratio,max_risk_score,network_risk_score
0,1,464,18,12,6,17,115123.22,7,4,0.411765,55.0,236.5
1,2,237,36,28,8,35,283657.03,6,3,0.171429,65.0,218.5
2,3,323,49,35,14,48,615137.04,9,2,0.187500,45.0,214.5
3,4,375,48,35,13,47,434789.05,11,2,0.234043,45.0,212.5
4,5,1680,12,8,4,11,83294.80,6,3,0.545455,65.0,209.5
5,6,925,14,11,3,13,171224.76,4,3,0.307692,68.0,193.0
6,7,250,31,24,7,30,204765.58,5,3,0.166667,55.0,188.5
7,8,35,65,45,20,64,728044.09,9,2,0.140625,45.0,182.5
8,9,290,32,23,9,31,396132.68,5,3,0.161290,48.0,179.0
9,10,353,13,10,3,12,102371.04,4,3,0.333333,55.0,174.5


In [69]:
# Create final suspicious network cluster table

suspicious_networks_final = (
    suspicious_networks[
        [
            "network_rank",
            "component_id",
            "node_count",
            "user_count",
            "merchant_count",
            "transaction_count",
            "transaction_value",
            "chargeback_transactions",
            "fraud_chargeback_transactions",
            "chargeback_ratio",
            "max_risk_score",
            "identity_anomaly_transactions",
            "repeated_chargeback_user_transactions",
            "high_chargeback_merchant_transactions",
            "network_risk_score"
        ]
    ]
    .head(50)
    .copy()
)

print("Final suspicious network table created.")
print("Rows:", len(suspicious_networks_final))
print("Columns:", len(suspicious_networks_final.columns))

display(suspicious_networks_final.head(20))

Final suspicious network table created.
Rows: 50
Columns: 15


,network_rank,component_id,node_count,user_count,merchant_count,transaction_count,transaction_value,chargeback_transactions,fraud_chargeback_transactions,chargeback_ratio,max_risk_score,identity_anomaly_transactions,repeated_chargeback_user_transactions,high_chargeback_merchant_transactions,network_risk_score
0,1,464,18,12,6,17,115123.22,7,4,0.411765,55.0,0.0,2,3,236.5
1,2,237,36,28,8,35,283657.03,6,3,0.171429,65.0,1.0,2,7,218.5
2,3,323,49,35,14,48,615137.04,9,2,0.187500,45.0,2.0,4,4,214.5
3,4,375,48,35,13,47,434789.05,11,2,0.234043,45.0,2.0,2,0,212.5
4,5,1680,12,8,4,11,83294.80,6,3,0.545455,65.0,0.0,3,4,209.5
5,6,925,14,11,3,13,171224.76,4,3,0.307692,68.0,2.0,2,3,193.0
6,7,250,31,24,7,30,204765.58,5,3,0.166667,55.0,0.0,0,7,188.5
7,8,35,65,45,20,64,728044.09,9,2,0.140625,45.0,0.0,2,0,182.5
8,9,290,32,23,9,31,396132.68,5,3,0.161290,48.0,3.0,0,0,179.0
9,10,353,13,10,3,12,102371.04,4,3,0.333333,55.0,1.0,0,4,174.5


In [70]:
# Save Step 34 network analysis outputs

suspicious_networks_final.to_csv(
    "../data/processed/suspicious_networks_top50.csv",
    index=False
)

component_risk.to_csv(
    "../data/processed/network_component_risk.csv",
    index=False
)

network_edges.to_csv(
    "../data/processed/network_edges.csv",
    index=False
)

print("Network analysis outputs saved successfully.")
print("suspicious_networks_top50.csv:", suspicious_networks_final.shape)
print("network_component_risk.csv:", component_risk.shape)
print("network_edges.csv:", network_edges.shape)

Network analysis outputs saved successfully.
suspicious_networks_top50.csv: (50, 15)
network_component_risk.csv: (5929, 15)
network_edges.csv: (20000, 12)


In [71]:
# Final validation of Step 34 network analysis

print("===== STEP 34 FINAL VALIDATION =====")

print("Network nodes:", G.number_of_nodes())
print("Network edges:", G.number_of_edges())

print("Total connected components:", len(connected_components))
print("Meaningful suspicious networks:", len(suspicious_networks))

print("\nTop suspicious network:")
display(
    suspicious_networks_final[
        [
            "network_rank",
            "component_id",
            "user_count",
            "merchant_count",
            "transaction_count",
            "chargeback_transactions",
            "fraud_chargeback_transactions",
            "chargeback_ratio",
            "network_risk_score"
        ]
    ].head(5)
)

print("\nStep 34 completed successfully.")

===== STEP 34 FINAL VALIDATION =====
Network nodes: 25929
Network edges: 20000
Total connected components: 5929
Meaningful suspicious networks: 4180

Top suspicious network:


,network_rank,component_id,user_count,merchant_count,transaction_count,chargeback_transactions,fraud_chargeback_transactions,chargeback_ratio,network_risk_score
0,1,464,12,6,17,7,4,0.411765,236.5
1,2,237,28,8,35,6,3,0.171429,218.5
2,3,323,35,14,48,9,2,0.187500,214.5
3,4,375,35,13,47,11,2,0.234043,212.5
4,5,1680,8,4,11,6,3,0.545455,209.5



Step 34 completed successfully.


In [72]:
# STEP 35 — Prepare final analytical datasets for Power BI

import pandas as pd
import numpy as np

# Load the final enriched transaction fact table
fact_transactions_enriched = pd.read_csv(
    "../data/processed/fact_transactions_enriched.csv"
)

print("Final fact table loaded successfully.")
print("Rows:", len(fact_transactions_enriched))
print("Columns:", len(fact_transactions_enriched.columns))
print("Unique transactions:", fact_transactions_enriched["txn_id"].nunique())
print(
    "Duplicate transaction IDs:",
    fact_transactions_enriched["txn_id"].duplicated().sum()
)

print("\nKey fields available:")
key_fields = [
    "txn_id",
    "user_id",
    "merchant_id",
    "amount_numeric",
    "status_clean",
    "merchant_category_final",
    "chargeback_flag",
    "fraud_chargeback_flag",
    "fraud_risk_score",
    "fraud_risk_band"
]

for col in key_fields:
    print(f"{col}: {'FOUND' if col in fact_transactions_enriched.columns else 'MISSING'}")

Final fact table loaded successfully.
Rows: 20000
Columns: 92
Unique transactions: 20000
Duplicate transaction IDs: 0

Key fields available:
txn_id: FOUND
user_id: FOUND
merchant_id: FOUND
amount_numeric: FOUND
status_clean: FOUND
merchant_category_final: MISSING
chargeback_flag: FOUND
fraud_chargeback_flag: FOUND
fraud_risk_score: MISSING
fraud_risk_band: MISSING


In [73]:
# STEP 35 — Restore analytical fields after reload

# --------------------------------------------------
# 1. Restore merchant_category_final
# --------------------------------------------------

if "merchant_category_final" not in fact_transactions_enriched.columns:

    category_source = None

    # Use the standardized category already present in the fact table
    for col in [
        "merchant_category",
        "merchant_category_clean",
        "category_clean"
    ]:
        if col in fact_transactions_enriched.columns:
            category_source = col
            break

    if category_source is None:
        raise KeyError(
            "No merchant category source column found in fact_transactions_enriched."
        )

    fact_transactions_enriched["merchant_category_final"] = (
        fact_transactions_enriched[category_source]
        .astype("string")
        .str.strip()
        .str.upper()
    )

    # Consolidate category variants used in Step 31
    category_map = {
        "BOOK STORE": "BOOKS",
        "BOOKS_STATIONERY": "BOOKS",
        "DEPARTMENT STORES": "DEPARTMENT STORE",
        "DEPT_STORE": "DEPARTMENT STORE",
        "GROCERIES": "GROCERY",
        "GROCERY STORES": "GROCERY",
        "GROCERY_STORE": "GROCERY",
        "KIRANA": "GROCERY",
        "HOTEL": "HOTEL_LODGING",
        "HOTELS": "HOTEL_LODGING",
        "HOSPITALITY": "HOTEL_LODGING",
        "PHARMACIES": "PHARMACY",
        "CHEMIST": "PHARMACY",
        "RESTAURANT": "FOOD_SERVICES",
        "RESTAURANTS": "FOOD_SERVICES",
        "EATING PLACE": "FOOD_SERVICES",
        "FOOD": "FOOD_SERVICES",
        "CLOTHING": "APPAREL",
        "CLOTHS": "APPAREL",
        "FASHION": "APPAREL",
        "GARMENTS": "APPAREL",
        "TRANSPORTATION": "TRANSPORT",
        "TRANSPRT": "TRANSPORT",
        "BUS/TAXI": "TRANSPORT",
        "MISC RETAIL": "MISC_RETAIL",
        "MISCELLANEOUS": "MISC_RETAIL",
        "RETAIL OTHER": "MISC_RETAIL"
    }

    fact_transactions_enriched["merchant_category_final"] = (
        fact_transactions_enriched["merchant_category_final"]
        .replace(category_map)
    )


# --------------------------------------------------
# 2. Restore fraud risk score
# --------------------------------------------------

risk_weights = {
    "fraud_chargeback_flag": 30,
    "chargeback_flag": 15,
    "strong_identity_anomaly_flag": 15,
    "repeated_chargeback_user_flag": 10,
    "high_chargeback_merchant_flag": 10,
    "high_risk_user_flag": 5,
    "kyc_rejected_transaction_flag": 5,
    "velocity_anomaly_flag": 5,
    "high_value_transaction_flag": 3,
    "merchant_status_risk_flag": 2
}

for col in risk_weights:
    fact_transactions_enriched[col] = pd.to_numeric(
        fact_transactions_enriched[col],
        errors="coerce"
    ).fillna(0)

fact_transactions_enriched["fraud_risk_score"] = 0

for col, weight in risk_weights.items():
    fact_transactions_enriched["fraud_risk_score"] += (
        fact_transactions_enriched[col] * weight
    )


# --------------------------------------------------
# 3. Restore fraud risk band
# --------------------------------------------------

fact_transactions_enriched["fraud_risk_band"] = pd.cut(
    fact_transactions_enriched["fraud_risk_score"],
    bins=[-1, 19, 39, 59, float("inf")],
    labels=["LOW", "MEDIUM", "HIGH", "CRITICAL"]
)


# --------------------------------------------------
# 4. Validation
# --------------------------------------------------

print("Analytical fields restored successfully.")

print("\nRequired fields:")
for col in [
    "merchant_category_final",
    "fraud_risk_score",
    "fraud_risk_band"
]:
    print(
        f"{col}:",
        "FOUND" if col in fact_transactions_enriched.columns else "MISSING"
    )

print("\nRisk band distribution:")
print(
    fact_transactions_enriched["fraud_risk_band"]
    .value_counts()
    .sort_index()
)

print("\nRisk score range:")
print(
    "Minimum:",
    fact_transactions_enriched["fraud_risk_score"].min()
)

print(
    "Maximum:",
    fact_transactions_enriched["fraud_risk_score"].max()
)

Analytical fields restored successfully.

Required fields:
merchant_category_final: FOUND
fraud_risk_score: FOUND
fraud_risk_band: FOUND

Risk band distribution:
fraud_risk_band
LOW         18553
MEDIUM        565
HIGH          788
CRITICAL       94
Name: count, dtype: int64

Risk score range:
Minimum: 0.0
Maximum: 85.0


In [74]:
# STEP 35 — Executive Overview KPI table

executive_kpis = pd.DataFrame({
    "metric": [
        "Total Transactions",
        "Total Transaction Value",
        "Average Transaction Value",
        "Successful Transactions",
        "Failed Transactions",
        "Pending Transactions",
        "Chargeback Transactions",
        "Fraud/Unauthorized Chargeback Transactions",
        "High-Value Transactions",
        "High-Risk Users",
        "KYC Rejected/Failed Transactions",
        "Risky Merchant Status Transactions",
        "HIGH Risk Transactions",
        "CRITICAL Risk Transactions",
        "HIGH + CRITICAL Risk Transactions"
    ],
    "value": [
        len(fact_transactions_enriched),
        fact_transactions_enriched["amount_numeric"].sum(),
        fact_transactions_enriched["amount_numeric"].mean(),
        fact_transactions_enriched["status_clean"].eq("SUCCESS").sum(),
        fact_transactions_enriched["status_clean"].isin(
            ["FAILED", "FAIL", "DECLINED"]
        ).sum(),
        fact_transactions_enriched["status_clean"].isin(
            ["PENDING", "PROCESSING", "INITIATED"]
        ).sum(),
        fact_transactions_enriched["chargeback_flag"].sum(),
        fact_transactions_enriched["fraud_chargeback_flag"].sum(),
        fact_transactions_enriched["high_value_transaction_flag"].sum(),
        fact_transactions_enriched["high_risk_user_flag"].sum(),
        fact_transactions_enriched["kyc_rejected_transaction_flag"].sum(),
        fact_transactions_enriched["merchant_status_risk_flag"].sum(),
        fact_transactions_enriched["fraud_risk_band"].eq("HIGH").sum(),
        fact_transactions_enriched["fraud_risk_band"].eq("CRITICAL").sum(),
        fact_transactions_enriched["fraud_risk_band"].isin(
            ["HIGH", "CRITICAL"]
        ).sum()
    ]
})

print("Executive KPI table created.")
print("Rows:", len(executive_kpis))
print("Columns:", len(executive_kpis.columns))

display(executive_kpis)

Executive KPI table created.
Rows: 15
Columns: 2


,metric,value
0,Total Transactions,2.000000e+04
1,Total Transaction Value,2.140577e+08
2,Average Transaction Value,1.189275e+04
3,Successful Transactions,1.705300e+04
4,Failed Transactions,1.955000e+03
5,Pending Transactions,9.920000e+02
6,Chargeback Transactions,2.451000e+03
7,Fraud/Unauthorized Chargeback Transactions,8.730000e+02
8,High-Value Transactions,8.790000e+02
9,High-Risk Users,6.860000e+02


In [75]:
# STEP 35 — Daily transaction trend table

daily_trend = (
    fact_transactions_enriched
    .groupby("transaction_date")
    .agg(
        transaction_count=("txn_id", "count"),
        transaction_value=("amount_numeric", "sum"),
        successful_transactions=(
            "status_clean",
            lambda x: (x == "SUCCESS").sum()
        ),
        failed_transactions=(
            "status_clean",
            lambda x: x.isin(["FAILED", "FAIL", "DECLINED"]).sum()
        ),
        pending_transactions=(
            "status_clean",
            lambda x: x.isin(["PENDING", "PROCESSING", "INITIATED"]).sum()
        ),
        chargeback_transactions=("chargeback_flag", "sum"),
        fraud_chargeback_transactions=("fraud_chargeback_flag", "sum"),
        high_risk_transactions=(
            "fraud_risk_band",
            lambda x: x.isin(["HIGH", "CRITICAL"]).sum()
        )
    )
    .reset_index()
    .sort_values("transaction_date")
)

daily_trend["chargeback_ratio"] = (
    daily_trend["chargeback_transactions"] /
    daily_trend["transaction_count"]
)

daily_trend["fraud_chargeback_ratio"] = (
    daily_trend["fraud_chargeback_transactions"] /
    daily_trend["transaction_count"]
)

print("Daily transaction trend table created.")
print("Rows:", len(daily_trend))
print("Columns:", len(daily_trend.columns))

display(daily_trend.head(10))

Daily transaction trend table created.
Rows: 117
Columns: 11


,transaction_date,transaction_count,transaction_value,successful_transactions,failed_transactions,pending_transactions,chargeback_transactions,fraud_chargeback_transactions,high_risk_transactions,chargeback_ratio,fraud_chargeback_ratio
0,2026-01-01,215,2333349.44,189,21,5,27,6,6,0.125581,0.027907
1,2026-01-02,198,2148265.31,169,20,9,20,8,8,0.101010,0.040404
2,2026-01-03,219,2310321.90,190,19,10,31,10,10,0.141553,0.045662
3,2026-01-04,180,2090475.91,153,17,10,20,3,3,0.111111,0.016667
4,2026-01-05,165,1607738.68,141,16,8,19,8,8,0.115152,0.048485
5,2026-01-06,182,2032604.09,144,29,9,19,8,8,0.104396,0.043956
6,2026-01-07,171,1791775.40,152,12,7,21,6,6,0.122807,0.035088
7,2026-01-08,178,1835813.65,149,19,10,17,7,7,0.095506,0.039326
8,2026-01-09,185,2010051.85,168,15,2,24,7,7,0.129730,0.037838
9,2026-01-10,197,2038445.10,169,21,7,19,7,7,0.096447,0.035533


In [77]:
# STEP 35 — Merchant Intelligence table
# Corrected for the persisted fact table structure

merchant_intelligence = (
    fact_transactions_enriched
    .groupby(
        [
            "merchant_id",
            "merchant_name",
            "merchant_category_final"
        ],
        dropna=False
    )
    .agg(
        transaction_count=("txn_id", "count"),
        transaction_value=("amount_numeric", "sum"),
        average_transaction_value=("amount_numeric", "mean"),
        chargeback_transactions=("chargeback_flag", "sum"),
        fraud_chargeback_transactions=("fraud_chargeback_flag", "sum"),
        high_risk_transactions=(
            "fraud_risk_band",
            lambda x: x.isin(["HIGH", "CRITICAL"]).sum()
        ),
        risky_status_transactions=(
            "merchant_status_risk_flag",
            "sum"
        ),
        high_chargeback_merchant_transactions=(
            "high_chargeback_merchant_flag",
            "sum"
        )
    )
    .reset_index()
)

# Chargeback ratio
merchant_intelligence["chargeback_ratio"] = (
    merchant_intelligence["chargeback_transactions"] /
    merchant_intelligence["transaction_count"]
)

# Fraud chargeback ratio
merchant_intelligence["fraud_chargeback_ratio"] = (
    merchant_intelligence["fraud_chargeback_transactions"] /
    merchant_intelligence["transaction_count"]
)

# High-risk transaction ratio
merchant_intelligence["high_risk_transaction_ratio"] = (
    merchant_intelligence["high_risk_transactions"] /
    merchant_intelligence["transaction_count"]
)

# Sort merchants by chargeback risk
merchant_intelligence = merchant_intelligence.sort_values(
    [
        "chargeback_ratio",
        "chargeback_transactions",
        "transaction_count"
    ],
    ascending=[False, False, False]
).reset_index(drop=True)

print("Merchant Intelligence table created.")
print("Rows:", len(merchant_intelligence))
print("Columns:", len(merchant_intelligence.columns))

display(merchant_intelligence.head(20))

Merchant Intelligence table created.
Rows: 8051
Columns: 14


,merchant_id,merchant_name,merchant_category_final,transaction_count,transaction_value,average_transaction_value,chargeback_transactions,fraud_chargeback_transactions,high_risk_transactions,risky_status_transactions,high_chargeback_merchant_transactions,chargeback_ratio,fraud_chargeback_ratio,high_risk_transaction_ratio
0,MCH1744,"Mani, Tara and Mane",TRAVEL,3,1470.19,1470.190000,3,2,2,0,3,1.0,0.666667,0.666667
1,MCH3549,Khalsa and Sons,FOOD_SERVICES,3,47166.97,15722.323333,3,2,2,0,3,1.0,0.666667,0.666667
2,MCH5278,"Sarraf, Peri and Ratti",MISC_RETAIL,3,45200.09,15066.696667,3,0,0,0,3,1.0,0.000000,0.000000
3,MCH1089,"Mann, Edwin and Misra",GROCERY,2,26353.47,13176.735000,2,0,0,0,0,1.0,0.000000,0.000000
4,MCH1186,NaN,<NA>,2,19490.03,9745.015000,2,0,0,0,0,1.0,0.000000,0.000000
5,MCH1527,NaN,<NA>,2,28718.08,14359.040000,2,1,1,0,0,1.0,0.500000,0.500000
6,MCH1760,NaN,<NA>,2,16465.87,8232.935000,2,0,0,0,0,1.0,0.000000,0.000000
7,MCH2048,NaN,<NA>,2,20606.44,10303.220000,2,1,1,0,0,1.0,0.500000,0.500000
8,MCH2204,NaN,<NA>,2,19604.76,9802.380000,2,0,0,0,0,1.0,0.000000,0.000000
9,MCH2529,Sheth-Pal,BOOKS,2,39310.36,19655.180000,2,1,1,0,0,1.0,0.500000,0.500000


In [78]:
# STEP 35 — Merchant Category Intelligence table

category_intelligence = (
    fact_transactions_enriched
    .groupby(
        "merchant_category_final",
        dropna=False
    )
    .agg(
        transaction_count=("txn_id", "count"),
        transaction_value=("amount_numeric", "sum"),
        average_transaction_value=("amount_numeric", "mean"),
        chargeback_transactions=("chargeback_flag", "sum"),
        fraud_chargeback_transactions=("fraud_chargeback_flag", "sum"),
        high_risk_transactions=(
            "fraud_risk_band",
            lambda x: x.isin(["HIGH", "CRITICAL"]).sum()
        ),
        risky_status_transactions=(
            "merchant_status_risk_flag",
            "sum"
        )
    )
    .reset_index()
)

# Chargeback-to-transaction ratio
category_intelligence["chargeback_ratio"] = (
    category_intelligence["chargeback_transactions"] /
    category_intelligence["transaction_count"]
)

# Fraud chargeback ratio
category_intelligence["fraud_chargeback_ratio"] = (
    category_intelligence["fraud_chargeback_transactions"] /
    category_intelligence["transaction_count"]
)

# High-risk transaction ratio
category_intelligence["high_risk_transaction_ratio"] = (
    category_intelligence["high_risk_transactions"] /
    category_intelligence["transaction_count"]
)

# Rank categories by chargeback ratio
category_intelligence["chargeback_ratio_rank"] = (
    category_intelligence["chargeback_ratio"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

category_intelligence = category_intelligence.sort_values(
    [
        "chargeback_ratio",
        "chargeback_transactions",
        "transaction_count"
    ],
    ascending=[False, False, False]
).reset_index(drop=True)

print("Merchant Category Intelligence table created.")
print("Rows:", len(category_intelligence))
print("Columns:", len(category_intelligence.columns))

display(category_intelligence)

Merchant Category Intelligence table created.
Rows: 19
Columns: 12


,merchant_category_final,transaction_count,transaction_value,average_transaction_value,chargeback_transactions,fraud_chargeback_transactions,high_risk_transactions,risky_status_transactions,chargeback_ratio,fraud_chargeback_ratio,high_risk_transaction_ratio,chargeback_ratio_rank
0,OTHER,195,2.217035e+06,12248.812541,35,11,11,30,0.179487,0.056410,0.056410,1
1,STATIONERY,256,2.979719e+06,13011.874541,36,12,12,16,0.140625,0.046875,0.046875,2
2,MEDICAL_STORE,208,2.128353e+06,11143.209634,29,11,11,17,0.139423,0.052885,0.052885,3
3,MEDICAL,158,1.774900e+06,12325.697917,22,8,8,17,0.139241,0.050633,0.050633,4
4,TELECOM,454,4.598871e+06,11468.505312,63,23,23,41,0.138767,0.050661,0.050661,5
5,RETAIL,289,3.040042e+06,12209.004498,40,12,13,20,0.138408,0.041522,0.044983,6
6,PHONE SERVICE,240,2.708886e+06,12717.773052,33,11,11,17,0.137500,0.045833,0.045833,7
7,TRANSPORT,816,8.571969e+06,11807.120193,111,36,37,50,0.136029,0.044118,0.045343,8
8,MISC_RETAIL,669,6.890983e+06,11542.684740,85,29,29,49,0.127055,0.043348,0.043348,9
9,TRAVEL,182,1.804988e+06,11423.974747,23,10,10,7,0.126374,0.054945,0.054945,10


In [79]:
# STEP 35 — User & KYC Risk Intelligence table

user_risk_intelligence = (
    fact_transactions_enriched
    .groupby(
        "user_id",
        dropna=False
    )
    .agg(
        transaction_count=("txn_id", "count"),
        transaction_value=("amount_numeric", "sum"),
        average_transaction_value=("amount_numeric", "mean"),
        chargeback_transactions=("chargeback_flag", "sum"),
        fraud_chargeback_transactions=("fraud_chargeback_flag", "sum"),
        high_risk_transactions=(
            "fraud_risk_band",
            lambda x: x.isin(["HIGH", "CRITICAL"]).sum()
        ),
        high_risk_user_flag=("high_risk_user_flag", "max"),
        repeated_chargeback_user_flag=(
            "repeated_chargeback_user_flag",
            "max"
        ),
        identity_anomaly_flag=(
            "strong_identity_anomaly_flag",
            "max"
        ),
        kyc_rejected_transactions=(
            "kyc_rejected_transaction_flag",
            "sum"
        )
    )
    .reset_index()
)

# Chargeback ratio
user_risk_intelligence["chargeback_ratio"] = (
    user_risk_intelligence["chargeback_transactions"] /
    user_risk_intelligence["transaction_count"]
)

# Fraud chargeback ratio
user_risk_intelligence["fraud_chargeback_ratio"] = (
    user_risk_intelligence["fraud_chargeback_transactions"] /
    user_risk_intelligence["transaction_count"]
)

# High-risk transaction ratio
user_risk_intelligence["high_risk_transaction_ratio"] = (
    user_risk_intelligence["high_risk_transactions"] /
    user_risk_intelligence["transaction_count"]
)

# Investigation flag
user_risk_intelligence["investigation_flag"] = (
    (user_risk_intelligence["high_risk_transactions"] > 0) |
    (user_risk_intelligence["fraud_chargeback_transactions"] > 0) |
    (user_risk_intelligence["repeated_chargeback_user_flag"] == 1) |
    (user_risk_intelligence["identity_anomaly_flag"] == 1)
).astype(int)

# Rank users by overall observable risk
user_risk_intelligence = user_risk_intelligence.sort_values(
    [
        "fraud_chargeback_transactions",
        "chargeback_transactions",
        "high_risk_transactions",
        "transaction_value"
    ],
    ascending=[False, False, False, False]
).reset_index(drop=True)

print("User & KYC Risk Intelligence table created.")
print("Rows:", len(user_risk_intelligence))
print("Columns:", len(user_risk_intelligence.columns))

display(user_risk_intelligence.head(20))

User & KYC Risk Intelligence table created.
Rows: 17878
Columns: 15


,user_id,transaction_count,transaction_value,average_transaction_value,chargeback_transactions,fraud_chargeback_transactions,high_risk_transactions,high_risk_user_flag,repeated_chargeback_user_flag,identity_anomaly_flag,kyc_rejected_transactions,chargeback_ratio,fraud_chargeback_ratio,high_risk_transaction_ratio,investigation_flag
0,USR51423,2,37974.69,18987.345000,2,2,2,False,1,0.0,0,1.000000,1.000000,1.000000,1
1,USR51504,3,37382.67,12460.890000,2,2,2,False,1,0.0,0,0.666667,0.666667,0.666667,1
2,USR18442,3,22640.09,11320.045000,2,2,2,False,1,0.0,0,0.666667,0.666667,0.666667,1
3,USR58627,2,15533.31,7766.655000,2,2,2,False,1,0.0,0,1.000000,1.000000,1.000000,1
4,USR63885,2,12016.50,6008.250000,2,2,2,False,1,0.0,0,1.000000,1.000000,1.000000,1
5,USR18716,2,8561.77,4280.885000,2,2,2,True,1,0.0,0,1.000000,1.000000,1.000000,1
6,USR60842,3,50182.02,16727.340000,2,1,1,False,1,0.0,0,0.666667,0.333333,0.333333,1
7,USR68953,2,33146.02,16573.010000,2,1,1,False,1,0.0,0,1.000000,0.500000,0.500000,1
8,USR88654,2,27558.57,13779.285000,2,1,1,False,1,0.0,0,1.000000,0.500000,0.500000,1
9,USR97464,2,24841.76,12420.880000,2,1,1,False,1,0.0,0,1.000000,0.500000,0.500000,1


In [80]:
# STEP 35 — Chargeback & Dispute Analytics table

chargebacks_analytics = pd.read_csv(
    "../data/processed/chargebacks_clean.csv"
)

print("Chargeback dataset loaded.")
print("Rows:", len(chargebacks_analytics))
print("Columns:", len(chargebacks_analytics.columns))

print("\nAvailable columns:")
print(chargebacks_analytics.columns.tolist())

Chargeback dataset loaded.
Rows: 2800
Columns: 13

Available columns:
['complaint_id', 'txn_id', 'user_id', 'merchant_id', 'transaction_timestamp', 'reported_timestamp', 'disputed_amount', 'reason_code', 'complaint_text', 'resolution_status', 'bank_response_timestamp', 'severity', 'channel']


In [81]:
# STEP 35 — Chargeback & Dispute Analytics table

# Convert disputed amount to numeric
chargebacks_analytics["disputed_amount"] = pd.to_numeric(
    chargebacks_analytics["disputed_amount"],
    errors="coerce"
)

# Parse timestamps
chargebacks_analytics["transaction_timestamp"] = pd.to_datetime(
    chargebacks_analytics["transaction_timestamp"],
    errors="coerce"
)

chargebacks_analytics["reported_timestamp"] = pd.to_datetime(
    chargebacks_analytics["reported_timestamp"],
    errors="coerce"
)

chargebacks_analytics["bank_response_timestamp"] = pd.to_datetime(
    chargebacks_analytics["bank_response_timestamp"],
    errors="coerce"
)

# Calculate reporting delay in hours
chargebacks_analytics["reporting_delay_hours"] = (
    chargebacks_analytics["reported_timestamp"] -
    chargebacks_analytics["transaction_timestamp"]
).dt.total_seconds() / 3600

# Calculate bank response delay in hours
chargebacks_analytics["bank_response_delay_hours"] = (
    chargebacks_analytics["bank_response_timestamp"] -
    chargebacks_analytics["reported_timestamp"]
).dt.total_seconds() / 3600

# Create fraud/unauthorized indicator
chargebacks_analytics["fraud_unauthorized_flag"] = (
    chargebacks_analytics["reason_code"]
    .astype("string")
    .str.upper()
    .str.contains(
        "FRAUD|UNAUTHORIZED|UNAUTHORISED",
        na=False,
        regex=True
    )
).astype(int)

# Create reporting date
chargebacks_analytics["report_date"] = (
    chargebacks_analytics["reported_timestamp"]
    .dt.date
)

print("Chargeback & Dispute Analytics table prepared.")

print("\nRows:", len(chargebacks_analytics))
print("Columns:", len(chargebacks_analytics.columns))

print("\nTotal disputed amount:")
print(
    round(
        chargebacks_analytics["disputed_amount"].sum(),
        2
    )
)

print("\nFraud/Unauthorized chargebacks:")
print(
    chargebacks_analytics["fraud_unauthorized_flag"].sum()
)

print("\nMedian reporting delay (hours):")
print(
    round(
        chargebacks_analytics["reporting_delay_hours"].median(),
        2
    )
)

print("\nMedian bank response delay (hours):")
print(
    round(
        chargebacks_analytics["bank_response_delay_hours"].median(),
        2
    )
)

display(
    chargebacks_analytics[
        [
            "complaint_id",
            "txn_id",
            "merchant_id",
            "disputed_amount",
            "reason_code",
            "resolution_status",
            "severity",
            "channel",
            "reporting_delay_hours",
            "bank_response_delay_hours",
            "fraud_unauthorized_flag"
        ]
    ].head(10)
)

Chargeback & Dispute Analytics table prepared.

Rows: 2800
Columns: 17

Total disputed amount:
2860700.5

Fraud/Unauthorized chargebacks:
421

Median reporting delay (hours):
72.0

Median bank response delay (hours):
402.34


,complaint_id,txn_id,merchant_id,disputed_amount,reason_code,resolution_status,severity,channel,reporting_delay_hours,bank_response_delay_hours,fraud_unauthorized_flag
0,CBK0002082,TXN00004325,MCH1127,NaN,Merchant Not Delivered,CLOSED,Critical,ivr,96.0,219.319444,0
1,CBK0001941,TXN00003720,3835,414.69,login compromised,In Progress,H,Call Center,NaN,NaN,0
2,CBK0001799,TXN00012539,MCH3700,NaN,customer issue,OPEN,P4,IVR,24.0,NaN,0
3,CBK0002465,TXN00017802,MCH4534,1303.05,no service,Rejected,H,ivr,NaN,NaN,0
4,CBK0001870,TXN00015944,MCH1686,1459.42,merchant service issue,In Progress,Low,App,NaN,NaN,0
5,CBK0002663,TXN00009741,MCH9584,NaN,service failed,Closed,LOW,Branch,NaN,NaN,0
6,CBK0000782,TXN00015086,MCH6008,NaN,extra amount deducted,In Progress,M,chatbot,NaN,NaN,0
7,CBK0001838,TXN00014426,MCH4526,4193.52,Service Not Provided,OPEN,M,CHATBOT,NaN,NaN,0
8,CBK0001095,TXN00015316,MCH5980,930.83,dispute raised,Open,P2,Branch,NaN,NaN,0
9,CBK0002541,TXN00002822,MCH3992,NaN,UNAUTHORISED,OPEN,P2,ivr,NaN,NaN,1


In [82]:
# STEP 35 — Investigation Evidence table

investigation_evidence_final = fact_transactions_enriched[
    fact_transactions_enriched["fraud_risk_band"].isin(
        ["HIGH", "CRITICAL"]
    )
].copy()

# Investigation priority
def assign_investigation_priority(row):
    if row["fraud_risk_band"] == "CRITICAL":
        return "P1 - Immediate Review"
    if row["fraud_chargeback_flag"] == 1:
        return "P2 - High Priority"
    return "P3 - Review"

investigation_evidence_final["investigation_priority"] = (
    investigation_evidence_final.apply(
        assign_investigation_priority,
        axis=1
    )
)

# Human-readable risk reasons
def get_risk_reasons(row):
    reasons = []

    if row["fraud_chargeback_flag"] == 1:
        reasons.append("Fraud/Unauthorized Chargeback")

    if row["chargeback_flag"] == 1:
        reasons.append("Chargeback")

    if row["strong_identity_anomaly_flag"] == 1:
        reasons.append("Identity Anomaly")

    if row["repeated_chargeback_user_flag"] == 1:
        reasons.append("Repeated User Chargebacks")

    if row["high_chargeback_merchant_flag"] == 1:
        reasons.append("High-Chargeback Merchant")

    if row["high_risk_user_flag"] == 1:
        reasons.append("High-Risk User")

    if row["kyc_rejected_transaction_flag"] == 1:
        reasons.append("KYC Rejected/Failed")

    if row["velocity_anomaly_flag"] == 1:
        reasons.append("Velocity Anomaly")

    if row["high_value_transaction_flag"] == 1:
        reasons.append("High-Value Transaction")

    if row["merchant_status_risk_flag"] == 1:
        reasons.append("Risky Merchant Status")

    return ", ".join(reasons)

investigation_evidence_final["risk_reasons"] = (
    investigation_evidence_final.apply(
        get_risk_reasons,
        axis=1
    )
)

# Recommended action
def recommended_action(row):
    if row["fraud_risk_band"] == "CRITICAL":
        return "Immediate fraud investigation"

    if row["fraud_chargeback_flag"] == 1:
        return "Review transaction and chargeback evidence"

    if row["strong_identity_anomaly_flag"] == 1:
        return "Review KYC and identity records"

    return "Review supporting risk signals"

investigation_evidence_final["recommended_action"] = (
    investigation_evidence_final.apply(
        recommended_action,
        axis=1
    )
)

# Keep only dashboard/investigation fields
investigation_evidence_final = investigation_evidence_final[
    [
        "txn_id",
        "timestamp",
        "user_id",
        "merchant_id",
        "merchant_name",
        "merchant_category_final",
        "amount_numeric",
        "status_clean",
        "fraud_risk_score",
        "fraud_risk_band",
        "investigation_priority",
        "risk_reasons",
        "chargeback_flag",
        "fraud_chargeback_flag",
        "strong_identity_anomaly_flag",
        "repeated_chargeback_user_flag",
        "high_chargeback_merchant_flag",
        "high_risk_user_flag",
        "kyc_rejected_transaction_flag",
        "velocity_anomaly_flag",
        "high_value_transaction_flag",
        "merchant_status_risk_flag",
        "kyc_status",
        "risk_segment",
        "merchant_status",
        "kyc_match_flag",
        "merchant_match_flag",
        "recommended_action"
    ]
].copy()

print("Investigation Evidence table created.")
print("Rows:", len(investigation_evidence_final))
print("Columns:", len(investigation_evidence_final.columns))

print("\nRisk band distribution:")
print(
    investigation_evidence_final["fraud_risk_band"]
    .value_counts()
    .sort_index()
)

print("\nInvestigation priority distribution:")
print(
    investigation_evidence_final["investigation_priority"]
    .value_counts()
)

display(investigation_evidence_final.head(10))

Investigation Evidence table created.
Rows: 882
Columns: 28

Risk band distribution:
fraud_risk_band
LOW           0
MEDIUM        0
HIGH        788
CRITICAL     94
Name: count, dtype: int64

Investigation priority distribution:
investigation_priority
P2 - High Priority       779
P1 - Immediate Review     94
P3 - Review                9
Name: count, dtype: int64


,txn_id,timestamp,user_id,merchant_id,merchant_name,merchant_category_final,amount_numeric,status_clean,fraud_risk_score,fraud_risk_band,...,kyc_rejected_transaction_flag,velocity_anomaly_flag,high_value_transaction_flag,merchant_status_risk_flag,kyc_status,risk_segment,merchant_status,kyc_match_flag,merchant_match_flag,recommended_action
42,TXN00008304,2026-03-20 05:00:36,USR95674,MCH5401,"Sama, Ar0ra and Mem0n",BOOKS,5379.22,SUCCESS,45.0,HIGH,...,False,0,False,False,UNKNOWN,MEDIUM,ACTIVE,True,True,Review transaction and chargeback evidence
46,TXN00017871,2026-03-23 02:31:39,USR84134,MCH7932,SETHI LTD,TELECOM,1129.96,FAIL,55.0,HIGH,...,False,0,False,False,NaN,NaN,Active,False,True,Review transaction and chargeback evidence
60,TXN00012810,2026-01-28 00:00:00,USR16089,MCH3478,NaN,<NA>,14920.61,SUCCESS,65.0,CRITICAL,...,False,0,False,False,APPROVED,LOW,NaN,True,False,Immediate fraud investigation
63,TXN00014556,2026-01-15 17:25:05,USR81688,MCH2222,"Mani, Ravel and Agrawal",APPAREL,10183.23,SUCCESS,45.0,HIGH,...,False,0,False,False,VERIFIED,LOW,ACTIVE,True,True,Review transaction and chargeback evidence
66,TXN00016736,2026-02-13 10:42:16,USR86671,MCH1318,GUPTA LLC,TELECOM,22925.42,SUCCESS,45.0,HIGH,...,False,0,False,False,NaN,NaN,Active,False,True,Review transaction and chargeback evidence
84,TXN00013883,2026-01-31 20:52:51,USR66862,MCH5569,"Thaman, Kar and Dass",MISC_RETAIL,16368.07,SUCCESS,45.0,HIGH,...,False,0,False,False,NaN,NaN,I,False,True,Review transaction and chargeback evidence
157,TXN00010462,2026-01-10 06:57:34,USR74619,MCH7994,NaN,<NA>,13136.01,SUCCESS,45.0,HIGH,...,False,0,False,False,NaN,NaN,NaN,False,False,Review transaction and chargeback evidence
222,TXN00000132,2026-01-31 15:03:31,USR43994,MCH5882,NaN,<NA>,5560.47,SUCCESS,45.0,HIGH,...,False,0,False,False,VERIFIED,LOW,NaN,True,False,Review transaction and chargeback evidence
258,TXN00003304,2026-01-28 22:13:38,USR58932,MCH3308,Misra Inc,RETAIL,8035.84,INITIATED,45.0,HIGH,...,False,0,False,False,NaN,NaN,A,False,True,Review transaction and chargeback evidence
292,TXN00000045,2026-02-03 00:50:28,USR81314,MCH2213,"Dash,SoniandPalla",HOTEL_LODGING,6476.91,SUCCESS,45.0,HIGH,...,False,0,False,False,APPROVED,LOW,ACTIVE,True,True,Review transaction and chargeback evidence


In [83]:
# STEP 35 — Save Power BI-ready analytical datasets

# 1. Executive KPI table
executive_kpis.to_csv(
    "../data/processed/powerbi_executive_kpis.csv",
    index=False
)

# 2. Daily transaction trend
daily_trend.to_csv(
    "../data/processed/powerbi_daily_trend.csv",
    index=False
)

# 3. Merchant intelligence
merchant_intelligence.to_csv(
    "../data/processed/powerbi_merchant_intelligence.csv",
    index=False
)

# 4. Merchant category intelligence
category_intelligence.to_csv(
    "../data/processed/powerbi_category_intelligence.csv",
    index=False
)

# 5. User/KYC risk intelligence
user_risk_intelligence.to_csv(
    "../data/processed/powerbi_user_risk_intelligence.csv",
    index=False
)

# 6. Chargeback/dispute analytics
chargebacks_analytics.to_csv(
    "../data/processed/powerbi_chargeback_analytics.csv",
    index=False
)

# 7. Investigation evidence
investigation_evidence_final.to_csv(
    "../data/processed/powerbi_investigation_evidence.csv",
    index=False
)

print("===== POWER BI DATASETS SAVED =====")

powerbi_files = {
    "Executive KPIs": executive_kpis,
    "Daily Trend": daily_trend,
    "Merchant Intelligence": merchant_intelligence,
    "Category Intelligence": category_intelligence,
    "User/KYC Intelligence": user_risk_intelligence,
    "Chargeback Analytics": chargebacks_analytics,
    "Investigation Evidence": investigation_evidence_final
}

for name, df in powerbi_files.items():
    print(f"{name}: {df.shape[0]} rows × {df.shape[1]} columns")

print("\nAll Power BI-ready datasets saved successfully.")

===== POWER BI DATASETS SAVED =====
Executive KPIs: 15 rows × 2 columns
Daily Trend: 117 rows × 11 columns
Merchant Intelligence: 8051 rows × 14 columns
Category Intelligence: 19 rows × 12 columns
User/KYC Intelligence: 17878 rows × 15 columns
Chargeback Analytics: 2800 rows × 17 columns
Investigation Evidence: 882 rows × 28 columns

All Power BI-ready datasets saved successfully.


In [84]:
# STEP 35 — Save Power BI-ready network datasets

# Top 50 suspicious network clusters
suspicious_networks_final.to_csv(
    "../data/processed/powerbi_suspicious_networks_top50.csv",
    index=False
)

# Complete network component risk table
component_risk.to_csv(
    "../data/processed/powerbi_network_component_risk.csv",
    index=False
)

# Transaction-level network edges
network_edges.to_csv(
    "../data/processed/powerbi_network_edges.csv",
    index=False
)

print("===== NETWORK POWER BI DATASETS SAVED =====")

print(
    "Suspicious Networks Top 50:",
    suspicious_networks_final.shape
)

print(
    "Network Component Risk:",
    component_risk.shape
)

print(
    "Network Edges:",
    network_edges.shape
)

print("\nAll network Power BI datasets saved successfully.")

===== NETWORK POWER BI DATASETS SAVED =====
Suspicious Networks Top 50: (50, 15)
Network Component Risk: (5929, 15)
Network Edges: (20000, 12)

All network Power BI datasets saved successfully.


In [85]:
# STEP 35 — FINAL PROJECT VALIDATION

print("=" * 60)
print("UPI FRAUD RING & MERCHANT ANALYTICS")
print("FINAL ANALYTICAL PIPELINE VALIDATION")
print("=" * 60)

# Core fact table
print("\n[1] CORE FACT TABLE")
print("Rows:", len(fact_transactions_enriched))
print("Columns:", len(fact_transactions_enriched.columns))
print("Unique TXNs:", fact_transactions_enriched["txn_id"].nunique())
print("Duplicate TXNs:", fact_transactions_enriched["txn_id"].duplicated().sum())

# Power BI datasets
print("\n[2] POWER BI DATASETS")

final_datasets = {
    "Executive KPIs": executive_kpis,
    "Daily Trend": daily_trend,
    "Merchant Intelligence": merchant_intelligence,
    "Category Intelligence": category_intelligence,
    "User/KYC Intelligence": user_risk_intelligence,
    "Chargeback Analytics": chargebacks_analytics,
    "Investigation Evidence": investigation_evidence_final,
    "Suspicious Networks": suspicious_networks_final,
    "Network Component Risk": component_risk,
    "Network Edges": network_edges
}

for name, df in final_datasets.items():
    print(f"{name:<30} {df.shape[0]:>6} rows × {df.shape[1]:>2} columns")

# Risk validation
print("\n[3] FRAUD RISK VALIDATION")

print(
    "LOW:",
    (fact_transactions_enriched["fraud_risk_band"] == "LOW").sum()
)

print(
    "MEDIUM:",
    (fact_transactions_enriched["fraud_risk_band"] == "MEDIUM").sum()
)

print(
    "HIGH:",
    (fact_transactions_enriched["fraud_risk_band"] == "HIGH").sum()
)

print(
    "CRITICAL:",
    (fact_transactions_enriched["fraud_risk_band"] == "CRITICAL").sum()
)

print(
    "HIGH + CRITICAL:",
    fact_transactions_enriched["fraud_risk_band"]
    .isin(["HIGH", "CRITICAL"])
    .sum()
)

# Network validation
print("\n[4] NETWORK VALIDATION")
print("Network nodes:", G.number_of_nodes())
print("Network edges:", G.number_of_edges())
print("Connected components:", len(connected_components))
print("Meaningful suspicious networks:", len(suspicious_networks))

print("\n" + "=" * 60)
print("STEP 35 COMPLETE — PROJECT ANALYTICS READY FOR POWER BI")
print("=" * 60)

UPI FRAUD RING & MERCHANT ANALYTICS
FINAL ANALYTICAL PIPELINE VALIDATION

[1] CORE FACT TABLE
Rows: 20000
Columns: 95
Unique TXNs: 20000
Duplicate TXNs: 0

[2] POWER BI DATASETS
Executive KPIs                     15 rows ×  2 columns
Daily Trend                       117 rows × 11 columns
Merchant Intelligence            8051 rows × 14 columns
Category Intelligence              19 rows × 12 columns
User/KYC Intelligence           17878 rows × 15 columns
Chargeback Analytics             2800 rows × 17 columns
Investigation Evidence            882 rows × 28 columns
Suspicious Networks                50 rows × 15 columns
Network Component Risk           5929 rows × 15 columns
Network Edges                   20000 rows × 12 columns

[3] FRAUD RISK VALIDATION
LOW: 18553
MEDIUM: 565
HIGH: 788
CRITICAL: 94
HIGH + CRITICAL: 882

[4] NETWORK VALIDATION
Network nodes: 25929
Network edges: 20000
Connected components: 5929
Meaningful suspicious networks: 4180

STEP 35 COMPLETE — PROJECT ANALYTICS 